In [1]:
# ==============================================================
# V7 — CLEAN LSTM LANGUAGE MODEL
# STEP 1 — CONFIGURATION & REPRODUCIBILITY
# ==============================================================
# IMPORTANT:
# This notebook is a fresh V7 implementation.
#
# VARIABLE NAMES ARE FIXED FOR THE ENTIRE NOTEBOOK.
#
# Do NOT rename:
#   train_tokens
#   val_tokens
#   test_tokens
#   word_to_id
#   id_to_word
#   X_train
#   y_train
#   X_val
#   y_val
#   X_test
#   y_test
#   lstm_v7
# ==============================================================

import os
import re
import random
import numpy as np
import tensorflow as tf

print("=" * 70)
print("V7 — CLEAN LSTM LANGUAGE MODEL")
print("=" * 70)

print("\nSTEP 1 — CONFIGURATION & REPRODUCIBILITY")
print("=" * 70)


# ==============================================================
# 1. RANDOM SEED
# ==============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


# ==============================================================
# 2. CORPUS CONFIGURATION
# ==============================================================

CORPUS_PATH = "/content/hamlet.txt"


# ==============================================================
# 3. DATA SPLIT CONFIGURATION
# ==============================================================

TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10


# ==============================================================
# 4. TOKENIZATION CONFIGURATION
# ==============================================================

# V7 CHANGE:
# Keep every word that appears at least once in the
# TRAINING corpus.
#
# V6 used:
#     MIN_WORD_FREQ = 2
#
# V7 uses:
#     MIN_WORD_FREQ = 1
#
# This is intended to reduce the excessive OOV problem
# observed in V6.

MIN_WORD_FREQ = 1


# ==============================================================
# 5. SEQUENCE CONFIGURATION
# ==============================================================

SEQ_LEN = 20


# ==============================================================
# 6. MODEL CONFIGURATION
# ==============================================================

EMBEDDING_DIM = 128
LSTM_UNITS = 128
DROPOUT_RATE = 0.30

LEARNING_RATE = 0.001

BATCH_SIZE = 64
MAX_EPOCHS = 30


# ==============================================================
# 7. SPECIAL TOKENS
# ==============================================================

PAD_TOKEN = "<PAD>"
OOV_TOKEN = "<OOV>"

PAD_ID = 0
OOV_ID = 1


# ==============================================================
# 8. CHECKPOINT CONFIGURATION
# ==============================================================

CHECKPOINT_PATH = "/content/best_lstm_v7.keras"


# ==============================================================
# 9. REPRODUCIBILITY SETTINGS
# ==============================================================

os.environ["PYTHONHASHSEED"] = str(SEED)

try:
    tf.config.experimental.enable_op_determinism()
    DETERMINISTIC_OPS = True
except Exception:
    DETERMINISTIC_OPS = False


# ==============================================================
# 10. CONFIGURATION SUMMARY
# ==============================================================

print("\nCONFIGURATION")
print("-" * 70)

print(f"Corpus path           : {CORPUS_PATH}")

print(f"\nData split")
print(f"Training ratio        : {TRAIN_RATIO:.0%}")
print(f"Validation ratio      : {VAL_RATIO:.0%}")
print(f"Testing ratio         : {TEST_RATIO:.0%}")

print(f"\nVocabulary")
print(f"Minimum word frequency: {MIN_WORD_FREQ}")

print(f"\nSequence")
print(f"Sequence length       : {SEQ_LEN}")

print(f"\nModel")
print(f"Embedding dimension   : {EMBEDDING_DIM}")
print(f"LSTM units            : {LSTM_UNITS}")
print(f"Dropout               : {DROPOUT_RATE}")

print(f"\nTraining")
print(f"Learning rate         : {LEARNING_RATE}")
print(f"Batch size            : {BATCH_SIZE}")
print(f"Maximum epochs        : {MAX_EPOCHS}")

print(f"\nSpecial tokens")
print(f"{PAD_TOKEN:20} → {PAD_ID}")
print(f"{OOV_TOKEN:20} → {OOV_ID}")

print(f"\nCheckpoint")
print(f"Path                  : {CHECKPOINT_PATH}")

print(f"\nRandom seed")
print(f"Seed                  : {SEED}")
print(f"Deterministic ops     : {DETERMINISTIC_OPS}")


# ==============================================================
# 11. CONFIGURATION VALIDATION
# ==============================================================

print("\n" + "=" * 70)
print("CONFIGURATION VALIDATION")
print("=" * 70)

assert abs(
    TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0
) < 1e-9

assert SEQ_LEN > 0
assert MIN_WORD_FREQ >= 1

assert EMBEDDING_DIM > 0
assert LSTM_UNITS > 0
assert 0.0 <= DROPOUT_RATE < 1.0

assert LEARNING_RATE > 0
assert BATCH_SIZE > 0
assert MAX_EPOCHS > 0

assert PAD_ID == 0
assert OOV_ID == 1

assert PAD_TOKEN != OOV_TOKEN

print("\nAll configuration checks passed.")


# ==============================================================
# 12. CORPUS FILE VALIDATION
# ==============================================================

print("\n" + "=" * 70)
print("CORPUS FILE VALIDATION")
print("=" * 70)

if not os.path.exists(CORPUS_PATH):
    raise FileNotFoundError(
        f"\nHamlet corpus not found:\n{CORPUS_PATH}\n\n"
        "Please upload hamlet.txt to /content/."
    )

file_size = os.path.getsize(CORPUS_PATH)

print(f"\nCorpus file found     : YES")
print(f"File size             : {file_size:,} bytes")


# ==============================================================
# 13. INITIAL VARIABLE CONTRACT
# ==============================================================

# These variables will be created in later steps.
#
# They are listed here so the entire V7 notebook follows
# one consistent naming convention.

print("\n" + "=" * 70)
print("V7 VARIABLE CONTRACT")
print("=" * 70)

print(
    """
Corpus
------
raw_text

Text processing
---------------
clean_text
all_words

Dataset split
-------------
train_words
val_words
test_words

Vocabulary
----------
word_to_id
id_to_word
word_counts
VOCAB_SIZE

Tokenized data
--------------
train_tokens
val_tokens
test_tokens

Sequence datasets
-----------------
X_train
y_train
X_val
y_val
X_test
y_test

Model
-----
lstm_v7

Training
--------
history_v7
best_lstm_v7.keras

Predictions
-----------
test_probabilities
predicted_ids
actual_ids
"""
)


# ==============================================================
# FINAL STATUS
# ==============================================================

print("=" * 70)
print("✅ STEP 1 COMPLETED — V7 CONFIGURATION READY")
print("=" * 70)

print("\nV7 baseline changes from V6:")
print("1. Minimum word frequency changed: 2 → 1")
print("2. Vocabulary will be built only from training data")
print("3. Variable names are fixed for the entire notebook")
print("4. V7 checkpoint is isolated from V6")
print("5. Reproducibility settings are enabled")

print("\nNext step:")
print("STEP 2 — CORPUS LOADING")

V7 — CLEAN LSTM LANGUAGE MODEL

STEP 1 — CONFIGURATION & REPRODUCIBILITY

CONFIGURATION
----------------------------------------------------------------------
Corpus path           : /content/hamlet.txt

Data split
Training ratio        : 80%
Validation ratio      : 10%
Testing ratio         : 10%

Vocabulary
Minimum word frequency: 1

Sequence
Sequence length       : 20

Model
Embedding dimension   : 128
LSTM units            : 128
Dropout               : 0.3

Training
Learning rate         : 0.001
Batch size            : 64
Maximum epochs        : 30

Special tokens
<PAD>                → 0
<OOV>                → 1

Checkpoint
Path                  : /content/best_lstm_v7.keras

Random seed
Seed                  : 42
Deterministic ops     : True

CONFIGURATION VALIDATION

All configuration checks passed.

CORPUS FILE VALIDATION

Corpus file found     : YES
File size             : 162,881 bytes

V7 VARIABLE CONTRACT

Corpus
------
raw_text

Text processing
---------------
clean_text
a

In [2]:
# ==============================================================
# V7 — STEP 2 — CORPUS LOADING
# ==============================================================
# IMPORTANT:
# Uses the exact variable name:
#     raw_text
#
# Do not rename this variable.
# ==============================================================

print("\n" + "=" * 70)
print("STEP 2 — CORPUS LOADING")
print("=" * 70)


# ==============================================================
# 1. LOAD CORPUS
# ==============================================================

with open(
    CORPUS_PATH,
    "r",
    encoding="utf-8"
) as file:

    raw_text = file.read()


# ==============================================================
# 2. BASIC CORPUS INFORMATION
# ==============================================================

character_count = len(raw_text)

line_count = len(
    raw_text.splitlines()
)

whitespace_words = len(
    raw_text.split()
)


print("\nCORPUS INFORMATION")
print("-" * 70)

print(f"File                : {CORPUS_PATH}")
print(f"Characters          : {character_count:,}")
print(f"Lines               : {line_count:,}")
print(f"Whitespace words    : {whitespace_words:,}")


# ==============================================================
# 3. EMPTY CORPUS CHECK
# ==============================================================

if not raw_text.strip():

    raise ValueError(
        "The corpus file is empty."
    )


# ==============================================================
# 4. FIRST 500 CHARACTERS
# ==============================================================

print("\n" + "=" * 70)
print("FIRST 500 CHARACTERS")
print("=" * 70)

print(raw_text[:500])


# ==============================================================
# 5. LAST 500 CHARACTERS
# ==============================================================

print("\n" + "=" * 70)
print("LAST 500 CHARACTERS")
print("=" * 70)

print(raw_text[-500:])


# ==============================================================
# 6. CORPUS SANITY CHECKS
# ==============================================================

print("\n" + "=" * 70)
print("CORPUS SANITY CHECKS")
print("=" * 70)

assert isinstance(
    raw_text,
    str
)

assert len(raw_text) > 0

assert len(raw_text.split()) > 0

print("\nraw_text type        : VALID")
print("Corpus not empty     : VALID")
print("Whitespace words     : VALID")


# ==============================================================
# 7. CHARACTER DISTRIBUTION CHECK
# ==============================================================

newline_count = raw_text.count("\n")
space_count = raw_text.count(" ")
tab_count = raw_text.count("\t")


print("\n" + "=" * 70)
print("WHITESPACE ANALYSIS")
print("=" * 70)

print(f"\nNewline characters   : {newline_count:,}")
print(f"Space characters     : {space_count:,}")
print(f"Tab characters       : {tab_count:,}")


# ==============================================================
# FINAL STATUS
# ==============================================================

print("\n" + "=" * 70)
print("✅ STEP 2 COMPLETED — CORPUS LOADED SUCCESSFULLY")
print("=" * 70)

print(
    f"\nraw_text is ready."
)

print(
    f"Characters : {character_count:,}"
)

print(
    f"Words      : {whitespace_words:,}"
)

print("\nNext step:")
print("STEP 3 — TEXT CLEANING & TOKENIZATION")


STEP 2 — CORPUS LOADING

CORPUS INFORMATION
----------------------------------------------------------------------
File                : /content/hamlet.txt
Characters          : 162,881
Lines               : 4,922
Whitespace words    : 29,605

FIRST 500 CHARACTERS
[The Tragedie of Hamlet by William Shakespeare 1599]


Actus Primus. Scoena Prima.

Enter Barnardo and Francisco two Centinels.

  Barnardo. Who's there?
  Fran. Nay answer me: Stand & vnfold
your selfe

   Bar. Long liue the King

   Fran. Barnardo?
  Bar. He

   Fran. You come most carefully vpon your houre

   Bar. 'Tis now strook twelue, get thee to bed Francisco

   Fran. For this releefe much thankes: 'Tis bitter cold,
And I am sicke at heart

   Barn. Haue you had quiet Guard?
  Fran. Not

LAST 500 CHARACTERS
e
On plots, and errors happen

   For. Let foure Captaines
Beare Hamlet like a Soldier to the Stage,
For he was likely, had he beene put on
To haue prou'd most royally:
And for his passage,
The Souldiours Musick

In [3]:
# ==============================================================
# V7 — STEP 3 — TEXT CLEANING & TOKENIZATION
# ==============================================================
# IMPORTANT:
# This step creates:
#
#     clean_text
#     all_words
#
# These variable names are part of the V7 variable contract.
#
# V7 CHANGE:
# We use regex-based word tokenization instead of simple
# whitespace splitting.
#
# This separates punctuation from words.
#
# Example:
#
#     "lord?"  -> "lord"
#     "sir,"   -> "sir"
#     "ham."   -> "ham"
#
# Apostrophes inside words are preserved:
#
#     "who's" -> "who's"
#     "tis"   -> "tis"
#
# ==============================================================

import re

print("\n" + "=" * 70)
print("STEP 3 — TEXT CLEANING & TOKENIZATION")
print("=" * 70)


# ==============================================================
# 1. ORIGINAL CORPUS STATISTICS
# ==============================================================

original_characters = len(raw_text)
original_words = len(raw_text.split())

print("\nORIGINAL CORPUS")
print("-" * 70)

print(
    f"Characters          : "
    f"{original_characters:,}"
)

print(
    f"Whitespace words     : "
    f"{original_words:,}"
)


# ==============================================================
# 2. NORMALIZE TEXT CASE
# ==============================================================

clean_text = raw_text.lower()


# ==============================================================
# 3. NORMALIZE SPECIAL WHITESPACE
# ==============================================================

clean_text = re.sub(
    r"\s+",
    " ",
    clean_text
).strip()


# ==============================================================
# 4. TOKENIZE TEXT
# ==============================================================

# Tokenization rules:
#
# [a-z]+       → normal alphabetic words
#
# '[a-z]+      → contractions beginning with apostrophe
#
# [0-9]+       → numbers
#
# We intentionally do NOT keep punctuation attached to words.
#
# Examples:
#
# "hamlet."       → hamlet
# "lord?"         → lord
# "sir,"          → sir
# "who's"         → who's
# "tis"           → tis
# "1599"          → 1599
#
token_pattern = r"[a-z]+(?:'[a-z]+)?|[0-9]+"

all_words = re.findall(
    token_pattern,
    clean_text
)


# ==============================================================
# 5. REBUILD CLEAN TEXT
# ==============================================================

clean_text = " ".join(
    all_words
)


# ==============================================================
# 6. CLEANING RESULTS
# ==============================================================

clean_characters = len(clean_text)
clean_word_count = len(all_words)
unique_words = len(set(all_words))


print("\n" + "=" * 70)
print("CLEANING RESULTS")
print("-" * 70)

print(
    f"Original characters : "
    f"{original_characters:,}"
)

print(
    f"Clean characters    : "
    f"{clean_characters:,}"
)

print(
    f"Original words      : "
    f"{original_words:,}"
)

print(
    f"Clean words         : "
    f"{clean_word_count:,}"
)

print(
    f"Unique words        : "
    f"{unique_words:,}"
)


# ==============================================================
# 7. CLEANED TEXT SAMPLE
# ==============================================================

print("\n" + "=" * 70)
print("CLEANED TEXT SAMPLE")
print("-" * 70)

print(
    clean_text[:1000]
)


# ==============================================================
# 8. TOKENIZATION SAMPLE
# ==============================================================

print("\n" + "=" * 70)
print("TOKENIZATION SAMPLE")
print("-" * 70)

sample_text = """
Who's there?
Nay answer me: Stand & vnfold
your selfe.
Long liue the King.
'Tis bitter cold,
And I am sicke at heart.
"""

sample_tokens = re.findall(
    token_pattern,
    sample_text.lower()
)

print("\nOriginal sample:")
print(sample_text)

print("Tokens:")
print(sample_tokens)


# ==============================================================
# 9. PUNCTUATION CHECK
# ==============================================================

punctuation_attached_tokens = [
    word
    for word in all_words
    if re.search(
        r"[^a-z0-9']",
        word
    )
]


print("\n" + "=" * 70)
print("TOKEN VALIDATION")
print("-" * 70)

print(
    f"\nTotal tokens         : "
    f"{len(all_words):,}"
)

print(
    f"Unique tokens        : "
    f"{len(set(all_words)):,}"
)

print(
    f"Punctuation-attached : "
    f"{len(punctuation_attached_tokens):,}"
)


# ==============================================================
# 10. ASSERT TOKEN VALIDITY
# ==============================================================

assert len(all_words) > 0

assert all(
    re.fullmatch(
        r"[a-z]+(?:'[a-z]+)?|[0-9]+",
        word
    )
    for word in all_words
)


print("\nToken format validation : PASSED")


# ==============================================================
# 11. CHECK IMPORTANT WORDS
# ==============================================================

print("\n" + "=" * 70)
print("IMPORTANT TOKEN CHECK")
print("-" * 70)

check_words = [
    "the",
    "king",
    "hamlet",
    "love",
    "death",
    "selfe",
    "vnfold",
    "who's"
]

for word in check_words:

    count = all_words.count(word)

    print(
        f"{word:15} "
        f"→ count={count}"
    )


# ==============================================================
# 12. FREQUENCY PREVIEW
# ==============================================================

from collections import Counter

word_counts_preview = Counter(
    all_words
)

print("\n" + "=" * 70)
print("TOP 20 WORDS")
print("-" * 70)

for rank, (word, count) in enumerate(
    word_counts_preview.most_common(20),
    start=1
):

    print(
        f"{rank:2}. "
        f"{word:15} "
        f"frequency={count}"
    )


# ==============================================================
# 13. FINAL STATUS
# ==============================================================

print("\n" + "=" * 70)
print("STEP 3 SUMMARY")
print("=" * 70)

print(
    f"""
Original characters : {original_characters:,}
Clean characters    : {clean_characters:,}

Original words      : {original_words:,}
Clean words         : {clean_word_count:,}

Unique words        : {unique_words:,}

Tokenization        : REGEX
Case normalization  : LOWERCASE
Punctuation attached: {len(punctuation_attached_tokens):,}
"""
)

print("=" * 70)
print("✅ STEP 3 COMPLETED — TEXT CLEANING & TOKENIZATION SUCCESSFUL")
print("=" * 70)

print("\nVariables created:")
print("clean_text")
print("all_words")

print("\nNext step:")
print("STEP 4 — TRAIN / VALIDATION / TEST SPLIT")


STEP 3 — TEXT CLEANING & TOKENIZATION

ORIGINAL CORPUS
----------------------------------------------------------------------
Characters          : 162,881
Whitespace words     : 29,605

CLEANING RESULTS
----------------------------------------------------------------------
Original characters : 162,881
Clean characters    : 152,582
Original words      : 29,605
Clean words         : 29,699
Unique words        : 4,813

CLEANED TEXT SAMPLE
----------------------------------------------------------------------
the tragedie of hamlet by william shakespeare 1599 actus primus scoena prima enter barnardo and francisco two centinels barnardo who's there fran nay answer me stand vnfold your selfe bar long liue the king fran barnardo bar he fran you come most carefully vpon your houre bar tis now strook twelue get thee to bed francisco fran for this releefe much thankes tis bitter cold and i am sicke at heart barn haue you had quiet guard fran not a mouse stirring barn well goodnight if you do 

In [4]:
# ==============================================================
# V7 — CLEAN LSTM LANGUAGE MODEL
# STEP 4 — TRAIN / VALIDATION / TEST SPLIT
# ==============================================================
# IMPORTANT:
# - Uses the tokenized words created in STEP 3.
# - Split is performed AFTER cleaning/tokenization.
# - The split is sequential to preserve the original text order.
# - Variable names are part of the V7 contract and must not change.
# ==============================================================

import numpy as np

print("=" * 70)
print("STEP 4 — TRAIN / VALIDATION / TEST SPLIT")
print("=" * 70)


# ==============================================================
# 1. VERIFY REQUIRED VARIABLES
# ==============================================================

required_variables = [
    "raw_text",
    "clean_text",
    "all_words"
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise NameError(
        f"Missing variables: {missing_variables}\n\n"
        "Please run V7 Steps 1–3 before running Step 4."
    )

print("\nREQUIRED VARIABLES")
print("-" * 70)

for name in required_variables:
    print(f"{name:15} : FOUND")


# ==============================================================
# 2. DATASET INFORMATION
# ==============================================================

total_words = len(all_words)

print("\n" + "=" * 70)
print("DATASET INFORMATION")
print("=" * 70)

print(f"\nTotal tokenized words : {total_words:,}")


# ==============================================================
# 3. SPLIT CONFIGURATION
# ==============================================================

TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10

print("\n" + "=" * 70)
print("SPLIT CONFIGURATION")
print("=" * 70)

print(f"\nTraining ratio   : {TRAIN_RATIO:.0%}")
print(f"Validation ratio : {VAL_RATIO:.0%}")
print(f"Testing ratio    : {TEST_RATIO:.0%}")


# ==============================================================
# 4. VALIDATE SPLIT RATIOS
# ==============================================================

ratio_sum = (
    TRAIN_RATIO +
    VAL_RATIO +
    TEST_RATIO
)

assert np.isclose(
    ratio_sum,
    1.0
), "Split ratios must sum to 1.0"

assert TRAIN_RATIO > 0
assert VAL_RATIO > 0
assert TEST_RATIO > 0

print("\nSplit ratio validation : PASSED")


# ==============================================================
# 5. CALCULATE SPLIT INDICES
# ==============================================================
# We use sequential splitting.
#
# Example:
#
# 0% ---------------- 80% -------- 90% -------- 100%
# |     TRAIN          |    VAL     |    TEST     |
#
# This preserves the chronological/order structure of Hamlet.
# ==============================================================

train_end = int(
    total_words * TRAIN_RATIO
)

val_end = train_end + int(
    total_words * VAL_RATIO
)


# ==============================================================
# 6. CREATE DATA SPLITS
# ==============================================================

train_words = all_words[:train_end]

val_words = all_words[
    train_end:val_end
]

test_words = all_words[
    val_end:
]


# ==============================================================
# 7. SPLIT SIZE VALIDATION
# ==============================================================

actual_total = (
    len(train_words) +
    len(val_words) +
    len(test_words)
)

assert actual_total == total_words, (
    "Split sizes do not add up to total token count."
)

assert len(train_words) > 0
assert len(val_words) > 0
assert len(test_words) > 0


# ==============================================================
# 8. SPLIT RESULTS
# ==============================================================

print("\n" + "=" * 70)
print("SPLIT RESULTS")
print("=" * 70)

print(
    f"\nTraining words   : {len(train_words):,}"
)

print(
    f"Validation words : {len(val_words):,}"
)

print(
    f"Testing words    : {len(test_words):,}"
)

print(
    f"Total words      : {actual_total:,}"
)


# ==============================================================
# 9. ACTUAL SPLIT PERCENTAGES
# ==============================================================

train_percentage = (
    len(train_words) / total_words
) * 100

val_percentage = (
    len(val_words) / total_words
) * 100

test_percentage = (
    len(test_words) / total_words
) * 100

print("\n" + "=" * 70)
print("ACTUAL SPLIT PERCENTAGES")
print("=" * 70)

print(
    f"\nTraining   : {train_percentage:.2f}%"
)

print(
    f"Validation : {val_percentage:.2f}%"
)

print(
    f"Testing    : {test_percentage:.2f}%"
)


# ==============================================================
# 10. DATA LEAKAGE CHECK
# ==============================================================
# Because the split is sequential, the three datasets are
# contiguous sections of the original corpus.
#
# We verify that:
#
# train_words + val_words + test_words
# exactly reconstruct all_words.
# ==============================================================

reconstructed_words = (
    train_words +
    val_words +
    test_words
)

assert reconstructed_words == all_words, (
    "Data leakage / reconstruction check failed."
)

print("\n" + "=" * 70)
print("DATA SPLIT VALIDATION")
print("=" * 70)

print("\nSequential split : VALID")
print("Data reconstruction : VALID")
print("No words lost       : VALID")
print("No words duplicated : VALID")


# ==============================================================
# 11. BOUNDARY CHECK
# ==============================================================

print("\n" + "=" * 70)
print("SPLIT BOUNDARY CHECK")
print("=" * 70)

print("\nLast training words:")
print(
    " ".join(train_words[-10:])
)

print("\nFirst validation words:")
print(
    " ".join(val_words[:10])
)

print("\nLast validation words:")
print(
    " ".join(val_words[-10:])
)

print("\nFirst testing words:")
print(
    " ".join(test_words[:10])
)


# ==============================================================
# 12. VARIABLE TYPE VALIDATION
# ==============================================================

print("\n" + "=" * 70)
print("VARIABLE VALIDATION")
print("=" * 70)

assert isinstance(
    train_words,
    list
)

assert isinstance(
    val_words,
    list
)

assert isinstance(
    test_words,
    list
)

assert all(
    isinstance(word, str)
    for word in train_words
)

assert all(
    isinstance(word, str)
    for word in val_words
)

assert all(
    isinstance(word, str)
    for word in test_words
)

print("\ntrain_words : VALID")
print("val_words   : VALID")
print("test_words  : VALID")


# ==============================================================
# 13. FINAL STEP 4 SUMMARY
# ==============================================================

print("\n" + "=" * 70)
print("STEP 4 SUMMARY")
print("=" * 70)

print(
    f"""
Total tokenized words : {total_words:,}

Training words        : {len(train_words):,}
Validation words      : {len(val_words):,}
Testing words         : {len(test_words):,}

Training ratio        : {train_percentage:.2f}%
Validation ratio      : {val_percentage:.2f}%
Testing ratio         : {test_percentage:.2f}%

Split method          : Sequential
Data leakage          : None detected
Reconstruction        : PASSED
"""
)

print("=" * 70)
print("✅ STEP 4 COMPLETED — DATASET SPLIT SUCCESSFUL")
print("=" * 70)

print("\nVariables created:")
print("train_words")
print("val_words")
print("test_words")

print("\nNext step:")
print("STEP 5 — VOCABULARY CONSTRUCTION")

STEP 4 — TRAIN / VALIDATION / TEST SPLIT

REQUIRED VARIABLES
----------------------------------------------------------------------
raw_text        : FOUND
clean_text      : FOUND
all_words       : FOUND

DATASET INFORMATION

Total tokenized words : 29,699

SPLIT CONFIGURATION

Training ratio   : 80%
Validation ratio : 10%
Testing ratio    : 10%

Split ratio validation : PASSED

SPLIT RESULTS

Training words   : 23,759
Validation words : 2,969
Testing words    : 2,971
Total words      : 29,699

ACTUAL SPLIT PERCENTAGES

Training   : 80.00%
Validation : 10.00%
Testing    : 10.00%

DATA SPLIT VALIDATION

Sequential split : VALID
Data reconstruction : VALID
No words lost       : VALID
No words duplicated : VALID

SPLIT BOUNDARY CHECK

Last training words:
what would you vndertake to show your selfe your fathers

First validation words:
sonne indeed more then in words laer to cut his

Last validation words:
monument an houre of quiet shortly shall we see till

First testing words:
then in 

In [5]:
# ==============================================================
# V7 — CLEAN LSTM LANGUAGE MODEL
# STEP 5 — VOCABULARY CONSTRUCTION
# ==============================================================
# IMPORTANT V7 RULES:
#
# 1. Vocabulary is built ONLY from train_words.
# 2. Minimum frequency = 1.
# 3. Every word appearing in training data is retained.
# 4. Validation/test words not seen during training become <OOV>.
# 5. PAD ID = 0.
# 6. OOV ID = 1.
# 7. Vocabulary IDs remain fixed for the rest of V7.
#
# VARIABLE CONTRACT:
#
# word_to_id
# id_to_word
# word_counts
# VOCAB_SIZE
# ==============================================================

from collections import Counter

print("=" * 70)
print("STEP 5 — VOCABULARY CONSTRUCTION")
print("=" * 70)


# ==============================================================
# 1. VERIFY REQUIRED VARIABLES
# ==============================================================

required_variables = [
    "train_words",
    "val_words",
    "test_words"
]

missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise NameError(
        f"Missing variables: {missing_variables}\n\n"
        "Please run V7 Steps 1–4 before running Step 5."
    )

print("\nREQUIRED VARIABLES")
print("-" * 70)

for name in required_variables:
    print(f"{name:15} : FOUND")


# ==============================================================
# 2. SPECIAL TOKEN CONFIGURATION
# ==============================================================

PAD_ID = 0
OOV_ID = 1

PAD_TOKEN = "<PAD>"
OOV_TOKEN = "<OOV>"

MIN_WORD_FREQUENCY = 1


# ==============================================================
# 3. COUNT TRAINING WORDS
# ==============================================================
# CRITICAL:
# We count ONLY train_words.
#
# Validation and testing data must not influence the vocabulary.
# ==============================================================

word_counts = Counter(train_words)

total_training_words = len(train_words)
unique_training_words = len(word_counts)


print("\n" + "=" * 70)
print("TRAINING CORPUS")
print("=" * 70)

print(
    f"\nTotal training words : "
    f"{total_training_words:,}"
)

print(
    f"Unique training words: "
    f"{unique_training_words:,}"
)


# ==============================================================
# 4. FREQUENCY FILTER
# ==============================================================

vocabulary_words = [
    word
    for word, count in word_counts.items()
    if count >= MIN_WORD_FREQUENCY
]

print("\n" + "=" * 70)
print("FREQUENCY FILTER")
print("=" * 70)

print(
    f"\nMinimum frequency : "
    f"{MIN_WORD_FREQUENCY}"
)

print(
    f"Words retained    : "
    f"{len(vocabulary_words):,}"
)


# ==============================================================
# 5. BUILD VOCABULARY
# ==============================================================
# IDs:
#
# 0 → <PAD>
# 1 → <OOV>
# 2+ → normal vocabulary words
#
# We sort vocabulary words by:
#
# 1. Frequency descending
# 2. Alphabetical order for ties
#
# This makes vocabulary construction deterministic.
# ==============================================================

vocabulary_words = sorted(
    vocabulary_words,
    key=lambda word: (-word_counts[word], word)
)


word_to_id = {
    PAD_TOKEN: PAD_ID,
    OOV_TOKEN: OOV_ID
}

id_to_word = {
    PAD_ID: PAD_TOKEN,
    OOV_ID: OOV_TOKEN
}


for index, word in enumerate(
    vocabulary_words,
    start=2
):

    word_to_id[word] = index
    id_to_word[index] = word


# ==============================================================
# 6. VOCABULARY SIZE
# ==============================================================

VOCAB_SIZE = len(word_to_id)

expected_vocab_size = (
    len(vocabulary_words) + 2
)


# ==============================================================
# 7. VOCABULARY VALIDATION
# ==============================================================

assert VOCAB_SIZE == expected_vocab_size

assert PAD_ID == 0
assert OOV_ID == 1

assert word_to_id[PAD_TOKEN] == PAD_ID
assert word_to_id[OOV_TOKEN] == OOV_ID

assert id_to_word[PAD_ID] == PAD_TOKEN
assert id_to_word[OOV_ID] == OOV_TOKEN

assert len(word_to_id) == len(id_to_word)

assert set(word_to_id.values()) == set(
    id_to_word.keys()
)

assert min(id_to_word.keys()) == 0
assert max(id_to_word.keys()) == VOCAB_SIZE - 1


print("\n" + "=" * 70)
print("VOCABULARY")
print("=" * 70)

print(
    f"\nVocabulary words : "
    f"{len(vocabulary_words):,}"
)

print(
    f"PAD ID           : "
    f"{PAD_ID}"
)

print(
    f"OOV ID           : "
    f"{OOV_ID}"
)

print(
    f"Vocabulary size   : "
    f"{VOCAB_SIZE:,}"
)

print(
    f"Expected size     : "
    f"{expected_vocab_size:,}"
)


# ==============================================================
# 8. SAMPLE WORD → ID → WORD
# ==============================================================

print("\n" + "=" * 70)
print("SAMPLE WORD → ID → WORD")
print("=" * 70)

sample_words = [
    "the",
    "king",
    "hamlet",
    "love",
    "death",
    "shakespeare",
    "selfe",
    "vnfold",
    "who's"
]

print()

for word in sample_words:

    token_id = word_to_id.get(
        word,
        OOV_ID
    )

    decoded_word = id_to_word.get(
        token_id,
        "<UNKNOWN>"
    )

    print(
        f"{word:15} "
        f"→ {token_id:4} "
        f"→ {decoded_word}"
    )


# ==============================================================
# 9. SAMPLE WORD FREQUENCIES
# ==============================================================

print("\n" + "=" * 70)
print("SAMPLE WORD FREQUENCIES")
print("=" * 70)

print()

for word in sample_words:

    frequency = word_counts.get(
        word,
        0
    )

    retained = (
        word in word_to_id
        and word not in [
            PAD_TOKEN,
            OOV_TOKEN
        ]
    )

    print(
        f"{word:15} "
        f"→ frequency={frequency:<4} "
        f"retained={retained}"
    )


# ==============================================================
# 10. TOP 20 TRAINING WORDS
# ==============================================================

print("\n" + "=" * 70)
print("TOP 20 TRAINING WORDS")
print("=" * 70)

print()

for rank, (word, frequency) in enumerate(
    word_counts.most_common(20),
    start=1
):

    retained = word in word_to_id

    print(
        f"{rank:2}. "
        f"{word:20} "
        f"frequency={frequency:<4} "
        f"retained={retained}"
    )


# ==============================================================
# 11. VOCABULARY ID RANGE
# ==============================================================

print("\n" + "=" * 70)
print("VOCABULARY ID RANGE")
print("=" * 70)

minimum_id = min(id_to_word.keys())
maximum_id = max(id_to_word.keys())

print(
    f"\nMinimum ID : {minimum_id}"
)

print(
    f"Maximum ID : {maximum_id}"
)

print(
    f"Expected   : 0 → {VOCAB_SIZE - 1}"
)

assert minimum_id == 0
assert maximum_id == VOCAB_SIZE - 1


# ==============================================================
# 12. DIRECT ENCODE → DECODE TEST
# ==============================================================

print("\n" + "=" * 70)
print("DIRECT ENCODE → DECODE TEST")
print("=" * 70)

print()

for word in sample_words:

    token_id = word_to_id.get(
        word,
        OOV_ID
    )

    decoded_word = id_to_word[token_id]

    print(
        f"{word:15} "
        f"→ ID: {token_id:4} "
        f"→ {decoded_word}"
    )


# ==============================================================
# 13. ROUND-TRIP VALIDATION
# ==============================================================
# Every ID must decode to a word, and that word must encode
# back to the same ID.
# ==============================================================

round_trip_errors = []

for token_id, word in id_to_word.items():

    encoded_id = word_to_id.get(
        word
    )

    if encoded_id != token_id:

        round_trip_errors.append(
            (
                token_id,
                word,
                encoded_id
            )
        )


print("\n" + "=" * 70)
print("ROUND-TRIP VALIDATION")
print("=" * 70)

print(
    f"\nTotal vocabulary entries : "
    f"{len(id_to_word):,}"
)

print(
    f"Round-trip errors        : "
    f"{len(round_trip_errors)}"
)

assert len(round_trip_errors) == 0


# ==============================================================
# 14. TRAINING VOCABULARY COVERAGE
# ==============================================================
# Because MIN_WORD_FREQUENCY = 1, every unique training word
# should be present in the vocabulary.
# ==============================================================

training_words_missing = [
    word
    for word in word_counts
    if word not in word_to_id
]

training_coverage = (
    1.0
    if unique_training_words == 0
    else (
        (unique_training_words - len(training_words_missing))
        / unique_training_words
    )
)

print("\n" + "=" * 70)
print("TRAINING VOCABULARY COVERAGE")
print("=" * 70)

print(
    f"\nUnique training words : "
    f"{unique_training_words:,}"
)

print(
    f"Words in vocabulary  : "
    f"{unique_training_words - len(training_words_missing):,}"
)

print(
    f"Missing training words: "
    f"{len(training_words_missing):,}"
)

print(
    f"Training coverage     : "
    f"{training_coverage * 100:.2f}%"
)

assert len(training_words_missing) == 0


# ==============================================================
# 15. VALIDATION / TEST UNSEEN-WORD ANALYSIS
# ==============================================================
# This is expected.
#
# A word that appears only in validation/test but never appeared
# in training cannot have its own vocabulary ID.
#
# Such words will become <OOV> during tokenization.
# ==============================================================

validation_unseen_words = set(
    val_words
) - set(
    word_counts.keys()
)

testing_unseen_words = set(
    test_words
) - set(
    word_counts.keys()
)


print("\n" + "=" * 70)
print("UNSEEN WORD ANALYSIS")
print("=" * 70)

print(
    f"\nValidation unique unseen words : "
    f"{len(validation_unseen_words):,}"
)

print(
    f"Testing unique unseen words    : "
    f"{len(testing_unseen_words):,}"
)

print(
    "\nThese unseen validation/test words "
    "will correctly map to <OOV>."
)


# ==============================================================
# 16. FINAL STEP 5 SUMMARY
# ==============================================================

print("\n" + "=" * 70)
print("STEP 5 SUMMARY")
print("=" * 70)

print(
    f"""
Total training words : {total_training_words:,}
Unique training words: {unique_training_words:,}

Minimum frequency    : {MIN_WORD_FREQUENCY}
Words retained       : {len(vocabulary_words):,}

Vocabulary words     : {len(vocabulary_words):,}
PAD ID               : {PAD_ID}
OOV ID               : {OOV_ID}
Vocabulary size      : {VOCAB_SIZE:,}

ID range             : {minimum_id} → {maximum_id}

Training coverage    : {training_coverage * 100:.2f}%

Round-trip errors     : {len(round_trip_errors)}

Vocabulary source    : TRAINING DATA ONLY
"""
)

print("=" * 70)
print("✅ STEP 5 COMPLETED — VOCABULARY CONSTRUCTION SUCCESSFUL")
print("=" * 70)

print("\nVariables created:")
print("word_to_id")
print("id_to_word")
print("word_counts")
print("VOCAB_SIZE")

print("\nNext step:")
print("STEP 6 — TOKENIZATION")

STEP 5 — VOCABULARY CONSTRUCTION

REQUIRED VARIABLES
----------------------------------------------------------------------
train_words     : FOUND
val_words       : FOUND
test_words      : FOUND

TRAINING CORPUS

Total training words : 23,759
Unique training words: 4,117

FREQUENCY FILTER

Minimum frequency : 1
Words retained    : 4,117

VOCABULARY

Vocabulary words : 4,117
PAD ID           : 0
OOV ID           : 1
Vocabulary size   : 4,119
Expected size     : 4,119

SAMPLE WORD → ID → WORD

the             →    2 → the
king            →   30 → king
hamlet          →   49 → hamlet
love            →    1 → <OOV>
death           →  137 → death
shakespeare     → 3549 → shakespeare
selfe           →   63 → selfe
vnfold          →  968 → vnfold
who's           → 1560 → who's

SAMPLE WORD FREQUENCIES

the             → frequency=770  retained=True
king            → frequency=143  retained=True
hamlet          → frequency=74   retained=True
love            → frequency=0    retained=False
dea

In [6]:
# ==============================================================
# V7 — CLEAN LSTM LANGUAGE MODEL
# STEP 6 — TOKENIZATION
# ==============================================================
# IMPORTANT:
#
# V7 VARIABLE CONTRACT:
#
# Vocabulary:
#   word_to_id
#   id_to_word
#   word_counts
#   VOCAB_SIZE
#
# Tokenized data:
#   train_tokens
#   val_tokens
#   test_tokens
#
# Rules:
#   1. Training vocabulary was created ONLY from train_words.
#   2. Known words receive their vocabulary ID.
#   3. Unknown validation/test words receive OOV_ID.
#   4. No vocabulary is modified in this step.
#   5. PAD is NOT inserted here.
#      Padding is unnecessary because every sequence will have
#      exactly SEQ_LEN tokens.
# ==============================================================

import numpy as np

print("=" * 70)
print("STEP 6 — TOKENIZATION")
print("=" * 70)


# ==============================================================
# 1. VERIFY REQUIRED VARIABLES
# ==============================================================

required_variables = [
    "train_words",
    "val_words",
    "test_words",
    "word_to_id",
    "id_to_word",
    "word_counts",
    "VOCAB_SIZE",
    "PAD_ID",
    "OOV_ID"
]

missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise NameError(
        f"Missing variables: {missing_variables}\n\n"
        "Please run V7 Steps 1–5 before running Step 6."
    )

print("\nREQUIRED VARIABLES")
print("-" * 70)

for name in required_variables:
    print(f"{name:15} : FOUND")


# ==============================================================
# 2. TOKENIZATION FUNCTION
# ==============================================================

def tokenize_words(words, vocabulary):
    """
    Convert words into integer token IDs.

    Known word:
        word → vocabulary ID

    Unknown word:
        word → OOV_ID

    The vocabulary is NOT modified.
    """

    return [
        vocabulary.get(
            word,
            OOV_ID
        )
        for word in words
    ]


# ==============================================================
# 3. TOKENIZE TRAINING DATA
# ==============================================================

train_tokens = tokenize_words(
    train_words,
    word_to_id
)


# ==============================================================
# 4. TOKENIZE VALIDATION DATA
# ==============================================================

val_tokens = tokenize_words(
    val_words,
    word_to_id
)


# ==============================================================
# 5. TOKENIZE TESTING DATA
# ==============================================================

test_tokens = tokenize_words(
    test_words,
    word_to_id
)


# ==============================================================
# 6. CONVERT TO NUMPY ARRAYS
# ==============================================================
# int32 is appropriate for Keras Embedding input.
# ==============================================================

train_tokens = np.asarray(
    train_tokens,
    dtype=np.int32
)

val_tokens = np.asarray(
    val_tokens,
    dtype=np.int32
)

test_tokens = np.asarray(
    test_tokens,
    dtype=np.int32
)


# ==============================================================
# 7. TOKENIZATION RESULTS
# ==============================================================

print("\n" + "=" * 70)
print("TOKENIZATION RESULTS")
print("=" * 70)

print(
    f"\nTraining     Tokens: "
    f"{len(train_tokens):,}"
)

print(
    f"Validation   Tokens: "
    f"{len(val_tokens):,}"
)

print(
    f"Testing      Tokens: "
    f"{len(test_tokens):,}"
)


# ==============================================================
# 8. VERIFY TOKEN COUNTS
# ==============================================================

assert len(train_tokens) == len(train_words)
assert len(val_tokens) == len(val_words)
assert len(test_tokens) == len(test_words)

print("\nToken count validation : PASSED")


# ==============================================================
# 9. OOV ANALYSIS
# ==============================================================

train_oov_count = int(
    np.sum(train_tokens == OOV_ID)
)

val_oov_count = int(
    np.sum(val_tokens == OOV_ID)
)

test_oov_count = int(
    np.sum(test_tokens == OOV_ID)
)


train_oov_percentage = (
    train_oov_count /
    len(train_tokens)
) * 100

val_oov_percentage = (
    val_oov_count /
    len(val_tokens)
) * 100

test_oov_percentage = (
    test_oov_count /
    len(test_tokens)
) * 100


print("\n" + "=" * 70)
print("OOV ANALYSIS")
print("=" * 70)

print(
    f"\nTraining     OOV: "
    f"{train_oov_count:,} "
    f"({train_oov_percentage:.2f}%)"
)

print(
    f"Validation   OOV: "
    f"{val_oov_count:,} "
    f"({val_oov_percentage:.2f}%)"
)

print(
    f"Testing      OOV: "
    f"{test_oov_count:,} "
    f"({test_oov_percentage:.2f}%)"
)


# ==============================================================
# 10. TRAINING OOV CHECK
# ==============================================================
# Because MIN_WORD_FREQUENCY = 1 and vocabulary was built from
# ALL unique training words, training OOV should be exactly 0.
# ==============================================================

assert train_oov_count == 0, (
    "Training data contains OOV tokens. "
    "This should not happen with MIN_WORD_FREQUENCY = 1."
)

print("\nTraining OOV check : PASSED")


# ==============================================================
# 11. TOKEN ID RANGE CHECK
# ==============================================================

valid_min_id = 0
valid_max_id = VOCAB_SIZE - 1

train_invalid = np.sum(
    (train_tokens < valid_min_id) |
    (train_tokens > valid_max_id)
)

val_invalid = np.sum(
    (val_tokens < valid_min_id) |
    (val_tokens > valid_max_id)
)

test_invalid = np.sum(
    (test_tokens < valid_min_id) |
    (test_tokens > valid_max_id)
)


print("\n" + "=" * 70)
print("TOKEN RANGE CHECK")
print("=" * 70)

print(
    f"\nTraining     Min ID: "
    f"{train_tokens.min():4} | "
    f"Max ID: {train_tokens.max():4} | "
    f"Invalid: {train_invalid}"
)

print(
    f"Validation   Min ID: "
    f"{val_tokens.min():4} | "
    f"Max ID: {val_tokens.max():4} | "
    f"Invalid: {val_invalid}"
)

print(
    f"Testing      Min ID: "
    f"{test_tokens.min():4} | "
    f"Max ID: {test_tokens.max():4} | "
    f"Invalid: {test_invalid}"
)

assert train_invalid == 0
assert val_invalid == 0
assert test_invalid == 0


# ==============================================================
# 12. DIRECT TOKENIZATION TEST
# ==============================================================

print("\n" + "=" * 70)
print("DIRECT TOKENIZATION TEST")
print("=" * 70)

sample_words = [
    "the",
    "king",
    "hamlet",
    "love",
    "death",
    "shakespeare",
    "selfe",
    "vnfold",
    "who's"
]

print()

for word in sample_words:

    token_id = word_to_id.get(
        word,
        OOV_ID
    )

    decoded_word = id_to_word.get(
        token_id,
        "<UNKNOWN>"
    )

    print(
        f"{word:15} "
        f"→ ID: {token_id:4} "
        f"→ {decoded_word}"
    )


# ==============================================================
# 13. ROUND-TRIP TOKENIZATION VALIDATION
# ==============================================================
# Test several known IDs and verify:
#
# ID → word → ID
#
# Special tokens are included.
# ==============================================================

test_ids = [
    PAD_ID,
    OOV_ID,
    2,
    3,
    10,
    30,
    49,
    137,
    968,
    1560,
    VOCAB_SIZE - 1
]

print("\n" + "=" * 70)
print("ROUND-TRIP TOKENIZATION VALIDATION")
print("=" * 70)

round_trip_errors = []

print()

for token_id in test_ids:

    word = id_to_word.get(
        token_id,
        "<UNKNOWN>"
    )

    encoded_id = word_to_id.get(
        word,
        OOV_ID
    )

    valid = (
        encoded_id == token_id
    )

    if not valid:
        round_trip_errors.append(
            (
                token_id,
                word,
                encoded_id
            )
        )

    status = "✓" if valid else "✗"

    print(
        f"ID {token_id:<5} → "
        f"{word:<20} → "
        f"ID {encoded_id:<5} {status}"
    )


assert len(round_trip_errors) == 0


# ==============================================================
# 14. SAMPLE TOKENIZED DATA
# ==============================================================

print("\n" + "=" * 70)
print("TOKENIZED DATA SAMPLE")
print("=" * 70)

sample_size = min(
    20,
    len(train_tokens)
)

print(
    "\nFirst training words:"
)

print(
    " ".join(
        train_words[:sample_size]
    )
)

print(
    "\nCorresponding token IDs:"
)

print(
    train_tokens[:sample_size].tolist()
)

print(
    "\nDecoded token IDs:"
)

decoded_sample = [
    id_to_word.get(
        int(token_id),
        "<UNKNOWN>"
    )
    for token_id in train_tokens[:sample_size]
]

print(
    " ".join(decoded_sample)
)


# ==============================================================
# 15. TOKENIZATION INTEGRITY CHECK
# ==============================================================

print("\n" + "=" * 70)
print("TOKENIZATION INTEGRITY CHECK")
print("=" * 70)

assert len(train_words) == len(train_tokens)
assert len(val_words) == len(val_tokens)
assert len(test_words) == len(test_tokens)

assert train_tokens.dtype == np.int32
assert val_tokens.dtype == np.int32
assert test_tokens.dtype == np.int32

print("\nTraining tokens   : VALID")
print("Validation tokens : VALID")
print("Testing tokens    : VALID")

print("\nToken dtype       : int32")
print("Token IDs         : VALID")
print("Vocabulary mapping: VALID")


# ==============================================================
# 16. FINAL STEP 6 SUMMARY
# ==============================================================

print("\n" + "=" * 70)
print("STEP 6 SUMMARY")
print("=" * 70)

print(
    f"""
Vocabulary size : {VOCAB_SIZE:,}

PAD ID          : {PAD_ID}
OOV ID          : {OOV_ID}

Training tokens : {len(train_tokens):,}
Validation      : {len(val_tokens):,}
Testing         : {len(test_tokens):,}

Training OOV    : {train_oov_count:,} ({train_oov_percentage:.2f}%)
Validation OOV  : {val_oov_count:,} ({val_oov_percentage:.2f}%)
Testing OOV     : {test_oov_count:,} ({test_oov_percentage:.2f}%)

Invalid IDs     :
    Training    : {train_invalid}
    Validation  : {val_invalid}
    Testing     : {test_invalid}

Round-trip errors : {len(round_trip_errors)}
"""
)

print("=" * 70)
print("✅ STEP 6 COMPLETED — TOKENIZATION SUCCESSFUL")
print("=" * 70)

print("\nVariables created:")
print("train_tokens")
print("val_tokens")
print("test_tokens")

print("\nNext step:")
print("STEP 7 — SEQUENCE DATASET CREATION")

STEP 6 — TOKENIZATION

REQUIRED VARIABLES
----------------------------------------------------------------------
train_words     : FOUND
val_words       : FOUND
test_words      : FOUND
word_to_id      : FOUND
id_to_word      : FOUND
word_counts     : FOUND
VOCAB_SIZE      : FOUND
PAD_ID          : FOUND
OOV_ID          : FOUND

TOKENIZATION RESULTS

Training     Tokens: 23,759
Validation   Tokens: 2,969
Testing      Tokens: 2,971

Token count validation : PASSED

OOV ANALYSIS

Training     OOV: 0 (0.00%)
Validation   OOV: 469 (15.80%)
Testing      OOV: 413 (13.90%)

Training OOV check : PASSED

TOKEN RANGE CHECK

Training     Min ID:    2 | Max ID: 4118 | Invalid: 0
Validation   Min ID:    1 | Max ID: 4110 | Invalid: 0
Testing      Min ID:    1 | Max ID: 4097 | Invalid: 0

DIRECT TOKENIZATION TEST

the             → ID:    2 → the
king            → ID:   30 → king
hamlet          → ID:   49 → hamlet
love            → ID:    1 → <OOV>
death           → ID:  137 → death
shakespeare     →

In [7]:
# ==============================================================
# STEP 7 — SEQUENCE DATASET CREATION
# ==============================================================
# V7 CLEAN LSTM LANGUAGE MODEL
#
# Goal:
# Convert tokenized text into supervised next-word prediction
# sequences.
#
# Example:
#
# Input (20 tokens):
# [the, tragedie, of, hamlet, ... , who's]
#
# Target:
# [there]
#
# The model learns:
#
# 20 previous words  --->  next word
#
# IMPORTANT:
# Train, validation, and test sequences are created separately.
# Therefore, no sequence can cross a dataset split boundary.
# ==============================================================

import numpy as np

print("=" * 70)
print("STEP 7 — SEQUENCE DATASET CREATION")
print("=" * 70)


# ==============================================================
# 1. VERIFY REQUIRED VARIABLES
# ==============================================================

required_variables = [
    "train_tokens",
    "val_tokens",
    "test_tokens",
    "word_to_id",
    "id_to_word",
    "VOCAB_SIZE",
    "PAD_ID",
    "OOV_ID",
    "SEQ_LEN"
]

missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise NameError(
        f"Missing variables: {missing_variables}\n\n"
        "Please run Steps 1–6 before running Step 7."
    )


print("\nREQUIRED VARIABLES")
print("-" * 70)

for name in required_variables:
    print(f"{name:15} : FOUND")


# ==============================================================
# 2. SEQUENCE CONFIGURATION
# ==============================================================

print("\n" + "=" * 70)
print("SEQUENCE CONFIGURATION")
print("=" * 70)

print(f"\nSequence length : {SEQ_LEN}")
print(f"Vocabulary size : {VOCAB_SIZE}")
print(f"PAD ID          : {PAD_ID}")
print(f"OOV ID           : {OOV_ID}")


# ==============================================================
# 3. VALIDATE SEQUENCE LENGTH
# ==============================================================

if not isinstance(SEQ_LEN, int):
    raise TypeError(
        f"SEQ_LEN must be an integer. "
        f"Found: {type(SEQ_LEN)}"
    )

if SEQ_LEN <= 0:
    raise ValueError(
        f"SEQ_LEN must be greater than 0. "
        f"Found: {SEQ_LEN}"
    )

print("\nSequence length validation : PASSED")


# ==============================================================
# 4. VALIDATE TOKEN ARRAYS
# ==============================================================

print("\n" + "=" * 70)
print("TOKEN ARRAY VALIDATION")
print("=" * 70)


token_arrays = {
    "Training": train_tokens,
    "Validation": val_tokens,
    "Testing": test_tokens
}

for name, tokens in token_arrays.items():

    tokens = np.asarray(tokens)

    print(
        f"\n{name:12} : "
        f"shape={tokens.shape}, "
        f"dtype={tokens.dtype}, "
        f"tokens={len(tokens):,}"
    )

    if tokens.ndim != 1:
        raise ValueError(
            f"{name} tokens must be 1-dimensional. "
            f"Found shape: {tokens.shape}"
        )

    if len(tokens) <= SEQ_LEN:
        raise ValueError(
            f"{name} token count ({len(tokens)}) must be "
            f"greater than SEQ_LEN ({SEQ_LEN})."
        )

print("\nToken array validation : PASSED")


# ==============================================================
# 5. TOKEN ID VALIDATION
# ==============================================================

print("\n" + "=" * 70)
print("TOKEN ID VALIDATION")
print("-" * 70)

for name, tokens in token_arrays.items():

    tokens = np.asarray(tokens)

    invalid_mask = (
        (tokens < 0) |
        (tokens >= VOCAB_SIZE)
    )

    invalid_count = int(
        np.sum(invalid_mask)
    )

    print(
        f"{name:12} invalid token IDs : "
        f"{invalid_count}"
    )

    if invalid_count > 0:

        invalid_ids = np.unique(
            tokens[invalid_mask]
        )

        raise ValueError(
            f"{name} contains invalid token IDs: "
            f"{invalid_ids[:20]}"
        )

print("\nToken ID validation : PASSED")


# ==============================================================
# 6. FUNCTION TO CREATE SEQUENCES
# ==============================================================

def create_sequences(tokens, seq_len):
    """
    Convert a token sequence into next-token prediction samples.

    Example:

    tokens = [A, B, C, D, E]
    seq_len = 3

    X:
        [A, B, C]
        [B, C, D]

    y:
        D
        E

    Therefore:

        X[i] = tokens[i : i + seq_len]
        y[i] = tokens[i + seq_len]
    """

    tokens = np.asarray(
        tokens,
        dtype=np.int32
    )

    number_of_sequences = (
        len(tokens) - seq_len
    )

    X = np.empty(
        (
            number_of_sequences,
            seq_len
        ),
        dtype=np.int32
    )

    y = np.empty(
        number_of_sequences,
        dtype=np.int32
    )

    for i in range(number_of_sequences):

        X[i] = tokens[
            i:i + seq_len
        ]

        y[i] = tokens[
            i + seq_len
        ]

    return X, y


# ==============================================================
# 7. CREATE TRAINING SEQUENCES
# ==============================================================

print("\n" + "=" * 70)
print("CREATING TRAINING SEQUENCES")
print("=" * 70)

X_train, y_train = create_sequences(
    train_tokens,
    SEQ_LEN
)

print(f"\nTraining tokens    : {len(train_tokens):,}")
print(f"Training sequences : {len(X_train):,}")

print(f"X_train shape      : {X_train.shape}")
print(f"y_train shape      : {y_train.shape}")


# ==============================================================
# 8. CREATE VALIDATION SEQUENCES
# ==============================================================

print("\n" + "=" * 70)
print("CREATING VALIDATION SEQUENCES")
print("=" * 70)

X_val, y_val = create_sequences(
    val_tokens,
    SEQ_LEN
)

print(f"\nValidation tokens    : {len(val_tokens):,}")
print(f"Validation sequences : {len(X_val):,}")

print(f"X_val shape          : {X_val.shape}")
print(f"y_val shape          : {y_val.shape}")


# ==============================================================
# 9. CREATE TEST SEQUENCES
# ==============================================================

print("\n" + "=" * 70)
print("CREATING TEST SEQUENCES")
print("=" * 70)

X_test, y_test = create_sequences(
    test_tokens,
    SEQ_LEN
)

print(f"\nTesting tokens    : {len(test_tokens):,}")
print(f"Testing sequences : {len(X_test):,}")

print(f"X_test shape      : {X_test.shape}")
print(f"y_test shape      : {y_test.shape}")


# ==============================================================
# 10. EXPECTED SAMPLE COUNT VALIDATION
# ==============================================================

print("\n" + "=" * 70)
print("SEQUENCE COUNT VALIDATION")
print("=" * 70)

expected_train_samples = (
    len(train_tokens) - SEQ_LEN
)

expected_val_samples = (
    len(val_tokens) - SEQ_LEN
)

expected_test_samples = (
    len(test_tokens) - SEQ_LEN
)

print(
    f"\nTraining   expected={expected_train_samples:,} "
    f"actual={len(X_train):,}"
)

print(
    f"Validation expected={expected_val_samples:,} "
    f"actual={len(X_val):,}"
)

print(
    f"Testing    expected={expected_test_samples:,} "
    f"actual={len(X_test):,}"
)

assert len(X_train) == expected_train_samples
assert len(X_val) == expected_val_samples
assert len(X_test) == expected_test_samples

print("\nSequence count validation : PASSED")


# ==============================================================
# 11. INPUT/TARGET SHAPE VALIDATION
# ==============================================================

print("\n" + "=" * 70)
print("INPUT / TARGET SHAPE VALIDATION")
print("=" * 70)

print(
    f"\nX_train : {X_train.shape}"
)

print(
    f"y_train : {y_train.shape}"
)

print(
    f"\nX_val   : {X_val.shape}"
)

print(
    f"y_val   : {y_val.shape}"
)

print(
    f"\nX_test  : {X_test.shape}"
)

print(
    f"y_test  : {y_test.shape}"
)


assert X_train.ndim == 2
assert X_val.ndim == 2
assert X_test.ndim == 2

assert y_train.ndim == 1
assert y_val.ndim == 1
assert y_test.ndim == 1

assert X_train.shape[1] == SEQ_LEN
assert X_val.shape[1] == SEQ_LEN
assert X_test.shape[1] == SEQ_LEN

assert len(X_train) == len(y_train)
assert len(X_val) == len(y_val)
assert len(X_test) == len(y_test)

print("\nInput/target shape validation : PASSED")


# ==============================================================
# 12. TARGET ALIGNMENT VALIDATION
# ==============================================================

print("\n" + "=" * 70)
print("TARGET ALIGNMENT VALIDATION")
print("=" * 70)

# The target of every sequence must be the token
# immediately following the input sequence.

train_alignment_errors = np.sum(
    y_train != train_tokens[SEQ_LEN:]
)

val_alignment_errors = np.sum(
    y_val != val_tokens[SEQ_LEN:]
)

test_alignment_errors = np.sum(
    y_test != test_tokens[SEQ_LEN:]
)

print(
    f"\nTraining alignment errors   : "
    f"{train_alignment_errors}"
)

print(
    f"Validation alignment errors : "
    f"{val_alignment_errors}"
)

print(
    f"Testing alignment errors    : "
    f"{test_alignment_errors}"
)

assert train_alignment_errors == 0
assert val_alignment_errors == 0
assert test_alignment_errors == 0

print("\nTarget alignment : PASSED")


# ==============================================================
# 13. TOKEN ID RANGE VALIDATION
# ==============================================================

print("\n" + "=" * 70)
print("SEQUENCE TOKEN RANGE VALIDATION")
print("=" * 70)


sequence_sets = {
    "Training X": X_train,
    "Training y": y_train,
    "Validation X": X_val,
    "Validation y": y_val,
    "Testing X": X_test,
    "Testing y": y_test
}


for name, array in sequence_sets.items():

    minimum_id = int(
        np.min(array)
    )

    maximum_id = int(
        np.max(array)
    )

    invalid_count = int(
        np.sum(
            (array < 0) |
            (array >= VOCAB_SIZE)
        )
    )

    print(
        f"{name:15} "
        f"Min={minimum_id:4} "
        f"Max={maximum_id:4} "
        f"Invalid={invalid_count}"
    )

    assert invalid_count == 0


print("\nSequence token range validation : PASSED")


# ==============================================================
# 14. DTYPE VALIDATION
# ==============================================================

print("\n" + "=" * 70)
print("DATA TYPE VALIDATION")
print("=" * 70)

print(
    f"\nX_train dtype : {X_train.dtype}"
)

print(
    f"y_train dtype : {y_train.dtype}"
)

print(
    f"X_val dtype   : {X_val.dtype}"
)

print(
    f"y_val dtype   : {y_val.dtype}"
)

print(
    f"X_test dtype  : {X_test.dtype}"
)

print(
    f"y_test dtype  : {y_test.dtype}"
)

assert X_train.dtype == np.int32
assert y_train.dtype == np.int32
assert X_val.dtype == np.int32
assert y_val.dtype == np.int32
assert X_test.dtype == np.int32
assert y_test.dtype == np.int32

print("\nData type validation : PASSED")


# ==============================================================
# 15. DECODE SAMPLE TRAINING SEQUENCE
# ==============================================================

print("\n" + "=" * 70)
print("SAMPLE TRAINING SEQUENCE")
print("=" * 70)

sample_index = 0

sample_input_ids = X_train[
    sample_index
]

sample_target_id = int(
    y_train[sample_index]
)

sample_input_words = [
    id_to_word.get(
        int(token_id),
        "<UNKNOWN>"
    )
    for token_id in sample_input_ids
]

sample_target_word = id_to_word.get(
    sample_target_id,
    "<UNKNOWN>"
)

print("\nInput token IDs:")
print(
    sample_input_ids.tolist()
)

print("\nInput words:")
print(
    " ".join(sample_input_words)
)

print(
    f"\nTarget ID   : {sample_target_id}"
)

print(
    f"Target word : {sample_target_word}"
)


# ==============================================================
# 16. VERIFY SAMPLE TARGET DIRECTLY
# ==============================================================

print("\n" + "=" * 70)
print("SAMPLE TARGET VERIFICATION")
print("=" * 70)

direct_target_id = int(
    train_tokens[
        sample_index + SEQ_LEN
    ]
)

print(
    f"\nTarget from y_train : "
    f"{sample_target_id}"
)

print(
    f"Direct target       : "
    f"{direct_target_id}"
)

assert sample_target_id == direct_target_id

print("\nSample target verification : PASSED")


# ==============================================================
# 17. MULTIPLE SAMPLE SEQUENCES
# ==============================================================

print("\n" + "=" * 70)
print("MULTIPLE SEQUENCE SAMPLES")
print("=" * 70)

sample_count = min(
    5,
    len(X_train)
)

for i in range(sample_count):

    input_words = [
        id_to_word.get(
            int(token_id),
            "<UNKNOWN>"
        )
        for token_id in X_train[i]
    ]

    target_word = id_to_word.get(
        int(y_train[i]),
        "<UNKNOWN>"
    )

    print("\n" + "-" * 70)

    print(f"Sample {i + 1}")

    print(
        "\nInput : "
        + " ".join(input_words)
    )

    print(
        f"Target: {target_word}"
    )


# ==============================================================
# 18. SPLIT BOUNDARY SAFETY CHECK
# ==============================================================

print("\n" + "=" * 70)
print("SPLIT BOUNDARY SAFETY CHECK")
print("=" * 70)

print(
    """
Sequences were created independently:

train_tokens → X_train / y_train
val_tokens   → X_val   / y_val
test_tokens  → X_test  / y_test

No sequence is allowed to use tokens from another split.
"""
)

# Verify the final training target comes from training data.
assert (
    y_train[-1]
    ==
    train_tokens[-1]
)

# Verify the final validation target comes from validation data.
assert (
    y_val[-1]
    ==
    val_tokens[-1]
)

# Verify the final test target comes from test data.
assert (
    y_test[-1]
    ==
    test_tokens[-1]
)

print("Training boundary   : SAFE")
print("Validation boundary : SAFE")
print("Testing boundary    : SAFE")

print("\nSplit boundary safety : PASSED")


# ==============================================================
# 19. DATASET MEMORY INFORMATION
# ==============================================================

print("\n" + "=" * 70)
print("DATASET MEMORY INFORMATION")
print("=" * 70)

print(
    f"\nX_train memory : "
    f"{X_train.nbytes / (1024 ** 2):.2f} MB"
)

print(
    f"y_train memory : "
    f"{y_train.nbytes / (1024 ** 2):.2f} MB"
)

print(
    f"X_val memory   : "
    f"{X_val.nbytes / (1024 ** 2):.2f} MB"
)

print(
    f"y_val memory   : "
    f"{y_val.nbytes / (1024 ** 2):.2f} MB"
)

print(
    f"X_test memory  : "
    f"{X_test.nbytes / (1024 ** 2):.2f} MB"
)

print(
    f"y_test memory  : "
    f"{y_test.nbytes / (1024 ** 2):.2f} MB"
)


# ==============================================================
# 20. FINAL DATASET VALIDATION
# ==============================================================

print("\n" + "=" * 70)
print("FINAL DATASET VALIDATION")
print("=" * 70)

assert len(X_train) > 0
assert len(X_val) > 0
assert len(X_test) > 0

assert X_train.shape[1] == SEQ_LEN
assert X_val.shape[1] == SEQ_LEN
assert X_test.shape[1] == SEQ_LEN

assert X_train.shape[0] == y_train.shape[0]
assert X_val.shape[0] == y_val.shape[0]
assert X_test.shape[0] == y_test.shape[0]

assert np.all(
    (X_train >= 0) &
    (X_train < VOCAB_SIZE)
)

assert np.all(
    (X_val >= 0) &
    (X_val < VOCAB_SIZE)
)

assert np.all(
    (X_test >= 0) &
    (X_test < VOCAB_SIZE)
)

assert np.all(
    (y_train >= 0) &
    (y_train < VOCAB_SIZE)
)

assert np.all(
    (y_val >= 0) &
    (y_val < VOCAB_SIZE)
)

assert np.all(
    (y_test >= 0) &
    (y_test < VOCAB_SIZE)
)

print("\nAll final dataset checks : PASSED")


# ==============================================================
# 21. FINAL SUMMARY
# ==============================================================

print("\n" + "=" * 70)
print("STEP 7 SUMMARY")
print("=" * 70)

print(
    f"""
Sequence length      : {SEQ_LEN}

Vocabulary size      : {VOCAB_SIZE}

Training tokens      : {len(train_tokens):,}
Training sequences   : {len(X_train):,}

Validation tokens    : {len(val_tokens):,}
Validation sequences : {len(X_val):,}

Testing tokens       : {len(test_tokens):,}
Testing sequences    : {len(X_test):,}

X_train shape        : {X_train.shape}
y_train shape        : {y_train.shape}

X_val shape          : {X_val.shape}
y_val shape          : {y_val.shape}

X_test shape         : {X_test.shape}
y_test shape         : {y_test.shape}

Data type            : int32

Sequence method      : Sliding window
Prediction task      : Next-token prediction
Split boundary       : Preserved
Data leakage         : None detected
Target alignment     : Valid
Token IDs            : Valid
"""
)

print("=" * 70)
print("✅ STEP 7 COMPLETED — SEQUENCE DATASET CREATION SUCCESSFUL")
print("=" * 70)

print(
    """
Variables created:

X_train
y_train
X_val
y_val
X_test
y_test

Next step:
STEP 8 — V7 DATASET PIPELINE / INPUT PIPELINE PREPARATION
"""
)

STEP 7 — SEQUENCE DATASET CREATION

REQUIRED VARIABLES
----------------------------------------------------------------------
train_tokens    : FOUND
val_tokens      : FOUND
test_tokens     : FOUND
word_to_id      : FOUND
id_to_word      : FOUND
VOCAB_SIZE      : FOUND
PAD_ID          : FOUND
OOV_ID          : FOUND
SEQ_LEN         : FOUND

SEQUENCE CONFIGURATION

Sequence length : 20
Vocabulary size : 4119
PAD ID          : 0
OOV ID           : 1

Sequence length validation : PASSED

TOKEN ARRAY VALIDATION

Training     : shape=(23759,), dtype=int32, tokens=23,759

Validation   : shape=(2969,), dtype=int32, tokens=2,969

Testing      : shape=(2971,), dtype=int32, tokens=2,971

Token array validation : PASSED

TOKEN ID VALIDATION
----------------------------------------------------------------------
Training     invalid token IDs : 0
Validation   invalid token IDs : 0
Testing      invalid token IDs : 0

Token ID validation : PASSED

CREATING TRAINING SEQUENCES

Training tokens    : 23,

In [8]:
# ==============================================================
# STEP 8 — V7 DATASET PIPELINE / INPUT PIPELINE PREPARATION
# ==============================================================
# IMPORTANT:
# This step does NOT modify the sequences.
#
# It converts the NumPy datasets into TensorFlow tf.data.Dataset
# pipelines for efficient model training and evaluation.
#
# TRAINING:
#     from_tensor_slices
#          ↓
#     shuffle
#          ↓
#     batch
#          ↓
#     prefetch
#
# VALIDATION:
#     from_tensor_slices
#          ↓
#     batch
#          ↓
#     prefetch
#
# TESTING:
#     from_tensor_slices
#          ↓
#     batch
#          ↓
#     prefetch
#
# No tokenization
# No sequence modification
# No data augmentation
# No data leakage
# ==============================================================

import numpy as np
import tensorflow as tf

print("=" * 70)
print("STEP 8 — V7 DATASET PIPELINE / INPUT PIPELINE PREPARATION")
print("=" * 70)


# ==============================================================
# 1. VERIFY REQUIRED VARIABLES
# ==============================================================

required_variables = [
    "X_train",
    "y_train",
    "X_val",
    "y_val",
    "X_test",
    "y_test",
    "VOCAB_SIZE",
    "PAD_ID",
    "OOV_ID",
    "SEQ_LEN",
    "word_to_id",
    "id_to_word"
]

missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise NameError(
        f"Missing variables: {missing_variables}\n\n"
        "Please run Steps 1–7 before running Step 8."
    )

print("\nREQUIRED VARIABLES")
print("-" * 70)

for name in required_variables:
    print(f"{name:15} : FOUND")


# ==============================================================
# 2. PIPELINE CONFIGURATION
# ==============================================================

# --------------------------------------------------------------
# Use existing configuration values if already defined.
# Otherwise create safe V7 defaults.
# --------------------------------------------------------------

if "BATCH_SIZE" not in globals():
    BATCH_SIZE = 64

if "SEED" not in globals():
    SEED = 42

if "SHUFFLE_BUFFER" not in globals():
    SHUFFLE_BUFFER = min(
        len(X_train),
        10000
    )


print("\n" + "=" * 70)
print("PIPELINE CONFIGURATION")
print("=" * 70)

print(f"\nSequence length : {SEQ_LEN}")
print(f"Vocabulary size : {VOCAB_SIZE}")
print(f"Batch size      : {BATCH_SIZE}")
print(f"Shuffle buffer  : {SHUFFLE_BUFFER}")
print(f"Random seed     : {SEED}")


# ==============================================================
# 3. CONFIGURATION VALIDATION
# ==============================================================

if BATCH_SIZE <= 0:
    raise ValueError(
        "BATCH_SIZE must be greater than 0."
    )

if SHUFFLE_BUFFER <= 0:
    raise ValueError(
        "SHUFFLE_BUFFER must be greater than 0."
    )

if SEQ_LEN <= 0:
    raise ValueError(
        "SEQ_LEN must be greater than 0."
    )

if VOCAB_SIZE <= 0:
    raise ValueError(
        "VOCAB_SIZE must be greater than 0."
    )

print("\nConfiguration validation : PASSED")


# ==============================================================
# 4. NUMPY DATA VALIDATION
# ==============================================================

print("\n" + "=" * 70)
print("NUMPY DATA VALIDATION")
print("=" * 70)

datasets = {
    "X_train": X_train,
    "y_train": y_train,
    "X_val": X_val,
    "y_val": y_val,
    "X_test": X_test,
    "y_test": y_test
}

for name, data in datasets.items():

    if not isinstance(data, np.ndarray):
        raise TypeError(
            f"{name} must be a NumPy array."
        )

    print(
        f"{name:10} : "
        f"shape={str(data.shape):18} "
        f"dtype={data.dtype}"
    )

print("\nNumPy data validation : PASSED")


# ==============================================================
# 5. SHAPE VALIDATION
# ==============================================================

print("\n" + "=" * 70)
print("SHAPE VALIDATION")
print("=" * 70)


# Input arrays must be 2-dimensional.
assert X_train.ndim == 2
assert X_val.ndim == 2
assert X_test.ndim == 2


# Target arrays must be 1-dimensional.
assert y_train.ndim == 1
assert y_val.ndim == 1
assert y_test.ndim == 1


# Every sequence must have SEQ_LEN tokens.
assert X_train.shape[1] == SEQ_LEN
assert X_val.shape[1] == SEQ_LEN
assert X_test.shape[1] == SEQ_LEN


# Every input must have exactly one target.
assert len(X_train) == len(y_train)
assert len(X_val) == len(y_val)
assert len(X_test) == len(y_test)


print("\nTraining:")
print(f"X_train : {X_train.shape}")
print(f"y_train : {y_train.shape}")

print("\nValidation:")
print(f"X_val   : {X_val.shape}")
print(f"y_val   : {y_val.shape}")

print("\nTesting:")
print(f"X_test  : {X_test.shape}")
print(f"y_test  : {y_test.shape}")

print("\nShape validation : PASSED")


# ==============================================================
# 6. TOKEN ID RANGE VALIDATION
# ==============================================================

print("\n" + "=" * 70)
print("TOKEN ID RANGE VALIDATION")
print("=" * 70)


def count_invalid_ids(X, y, vocab_size):

    invalid_x = np.sum(
        (X < 0) |
        (X >= vocab_size)
    )

    invalid_y = np.sum(
        (y < 0) |
        (y >= vocab_size)
    )

    return int(invalid_x), int(invalid_y)


train_invalid_x, train_invalid_y = count_invalid_ids(
    X_train,
    y_train,
    VOCAB_SIZE
)

val_invalid_x, val_invalid_y = count_invalid_ids(
    X_val,
    y_val,
    VOCAB_SIZE
)

test_invalid_x, test_invalid_y = count_invalid_ids(
    X_test,
    y_test,
    VOCAB_SIZE
)


print(
    f"\nTraining     X invalid : "
    f"{train_invalid_x}"
)

print(
    f"Training     y invalid : "
    f"{train_invalid_y}"
)

print(
    f"Validation   X invalid : "
    f"{val_invalid_x}"
)

print(
    f"Validation   y invalid : "
    f"{val_invalid_y}"
)

print(
    f"Testing      X invalid : "
    f"{test_invalid_x}"
)

print(
    f"Testing      y invalid : "
    f"{test_invalid_y}"
)


assert train_invalid_x == 0
assert train_invalid_y == 0

assert val_invalid_x == 0
assert val_invalid_y == 0

assert test_invalid_x == 0
assert test_invalid_y == 0


print("\nToken ID range validation : PASSED")


# ==============================================================
# 7. CREATE TRAINING DATASET
# ==============================================================

print("\n" + "=" * 70)
print("CREATING TRAINING PIPELINE")
print("=" * 70)


train_dataset = tf.data.Dataset.from_tensor_slices(
    (
        X_train,
        y_train
    )
)


print("\nBase training dataset : CREATED")


# Shuffle training data only.
#
# IMPORTANT:
# We shuffle complete (X, y) pairs.
# Therefore inputs and their corresponding targets remain aligned.

train_dataset = train_dataset.shuffle(
    buffer_size=SHUFFLE_BUFFER,
    seed=SEED,
    reshuffle_each_iteration=True
)


# Create batches.
train_dataset = train_dataset.batch(
    BATCH_SIZE,
    drop_remainder=False
)


# Prefetch batches while the model is training.
train_dataset = train_dataset.prefetch(
    tf.data.AUTOTUNE
)


print("Shuffle                : ENABLED")
print("Batching               : ENABLED")
print("Prefetch               : AUTOTUNE")
print("Drop remainder         : False")


# ==============================================================
# 8. CREATE VALIDATION DATASET
# ==============================================================

print("\n" + "=" * 70)
print("CREATING VALIDATION PIPELINE")
print("=" * 70)


val_dataset = tf.data.Dataset.from_tensor_slices(
    (
        X_val,
        y_val
    )
)


# DO NOT shuffle validation data.
val_dataset = val_dataset.batch(
    BATCH_SIZE,
    drop_remainder=False
)


val_dataset = val_dataset.prefetch(
    tf.data.AUTOTUNE
)


print("\nValidation dataset : CREATED")
print("Shuffle            : DISABLED")
print("Batching           : ENABLED")
print("Prefetch           : AUTOTUNE")
print("Drop remainder     : False")


# ==============================================================
# 9. CREATE TEST DATASET
# ==============================================================

print("\n" + "=" * 70)
print("CREATING TEST PIPELINE")
print("=" * 70)


test_dataset = tf.data.Dataset.from_tensor_slices(
    (
        X_test,
        y_test
    )
)


# DO NOT shuffle test data.
#
# Keeping test order deterministic is useful for:
# - prediction analysis
# - sample inspection
# - error analysis
# - reproducibility

test_dataset = test_dataset.batch(
    BATCH_SIZE,
    drop_remainder=False
)


test_dataset = test_dataset.prefetch(
    tf.data.AUTOTUNE
)


print("\nTest dataset : CREATED")
print("Shuffle      : DISABLED")
print("Batching     : ENABLED")
print("Prefetch     : AUTOTUNE")
print("Drop remainder : False")


# ==============================================================
# 10. BATCH COUNT VALIDATION
# ==============================================================

print("\n" + "=" * 70)
print("BATCH COUNT VALIDATION")
print("=" * 70)


expected_train_batches = int(
    np.ceil(
        len(X_train) / BATCH_SIZE
    )
)

expected_val_batches = int(
    np.ceil(
        len(X_val) / BATCH_SIZE
    )
)

expected_test_batches = int(
    np.ceil(
        len(X_test) / BATCH_SIZE
    )
)


actual_train_batches = sum(
    1
    for _ in train_dataset
)

actual_val_batches = sum(
    1
    for _ in val_dataset
)

actual_test_batches = sum(
    1
    for _ in test_dataset
)


print(
    f"\nTraining     expected={expected_train_batches} "
    f"actual={actual_train_batches}"
)

print(
    f"Validation   expected={expected_val_batches} "
    f"actual={actual_val_batches}"
)

print(
    f"Testing      expected={expected_test_batches} "
    f"actual={actual_test_batches}"
)


assert actual_train_batches == expected_train_batches
assert actual_val_batches == expected_val_batches
assert actual_test_batches == expected_test_batches


print("\nBatch count validation : PASSED")


# ==============================================================
# 11. TRAINING BATCH VALIDATION
# ==============================================================

print("\n" + "=" * 70)
print("TRAINING BATCH VALIDATION")
print("=" * 70)


train_batch_X, train_batch_y = next(
    iter(train_dataset)
)


train_batch_size = train_batch_X.shape[0]


print(
    f"\nBatch X shape : "
    f"{train_batch_X.shape}"
)

print(
    f"Batch y shape : "
    f"{train_batch_y.shape}"
)

print(
    f"Batch X dtype : "
    f"{train_batch_X.dtype}"
)

print(
    f"Batch y dtype : "
    f"{train_batch_y.dtype}"
)


assert train_batch_X.shape[1] == SEQ_LEN

assert train_batch_y.shape[0] == (
    train_batch_X.shape[0]
)


assert train_batch_size <= BATCH_SIZE


print("\nTraining batch shape : VALID")


# ==============================================================
# 12. VALIDATION BATCH VALIDATION
# ==============================================================

print("\n" + "=" * 70)
print("VALIDATION BATCH VALIDATION")
print("=" * 70)


val_batch_X, val_batch_y = next(
    iter(val_dataset)
)


print(
    f"\nBatch X shape : "
    f"{val_batch_X.shape}"
)

print(
    f"Batch y shape : "
    f"{val_batch_y.shape}"
)


assert val_batch_X.shape[1] == SEQ_LEN

assert val_batch_y.shape[0] == (
    val_batch_X.shape[0]
)

assert val_batch_X.shape[0] <= BATCH_SIZE


print("\nValidation batch shape : VALID")


# ==============================================================
# 13. TEST BATCH VALIDATION
# ==============================================================

print("\n" + "=" * 70)
print("TEST BATCH VALIDATION")
print("=" * 70)


test_batch_X, test_batch_y = next(
    iter(test_dataset)
)


print(
    f"\nBatch X shape : "
    f"{test_batch_X.shape}"
)

print(
    f"Batch y shape : "
    f"{test_batch_y.shape}"
)


assert test_batch_X.shape[1] == SEQ_LEN

assert test_batch_y.shape[0] == (
    test_batch_X.shape[0]
)

assert test_batch_X.shape[0] <= BATCH_SIZE


print("\nTest batch shape : VALID")


# ==============================================================
# 14. SAMPLE BATCH DECODE
# ==============================================================

print("\n" + "=" * 70)
print("SAMPLE BATCH DECODE")
print("=" * 70)


sample_input_ids = (
    train_batch_X[0]
    .numpy()
)

sample_target_id = int(
    train_batch_y[0]
    .numpy()
)


sample_words = [
    id_to_word.get(
        int(token_id),
        "<UNKNOWN>"
    )
    for token_id in sample_input_ids
]


sample_target_word = id_to_word.get(
    sample_target_id,
    "<UNKNOWN>"
)


print("\nSample input token IDs:")
print(
    sample_input_ids.tolist()
)


print("\nSample input words:")
print(
    " ".join(sample_words)
)


print(
    f"\nTarget ID   : "
    f"{sample_target_id}"
)


print(
    f"Target word : "
    f"{sample_target_word}"
)


# ==============================================================
# 15. DATASET SIZE VALIDATION
# ==============================================================

print("\n" + "=" * 70)
print("DATASET SIZE VALIDATION")
print("=" * 70)


def count_dataset_examples(dataset):

    total = 0

    for batch_X, batch_y in dataset:

        total += int(
            batch_X.shape[0]
        )

    return total


train_dataset_count = count_dataset_examples(
    train_dataset
)

val_dataset_count = count_dataset_examples(
    val_dataset
)

test_dataset_count = count_dataset_examples(
    test_dataset
)


print(
    f"\nTraining examples   : "
    f"{train_dataset_count:,}"
)

print(
    f"Expected training   : "
    f"{len(X_train):,}"
)


print(
    f"\nValidation examples : "
    f"{val_dataset_count:,}"
)

print(
    f"Expected validation : "
    f"{len(X_val):,}"
)


print(
    f"\nTesting examples    : "
    f"{test_dataset_count:,}"
)

print(
    f"Expected testing    : "
    f"{len(X_test):,}"
)


assert train_dataset_count == len(X_train)
assert val_dataset_count == len(X_val)
assert test_dataset_count == len(X_test)


print("\nDataset size validation : PASSED")


# ==============================================================
# 16. PIPELINE TYPE VALIDATION
# ==============================================================

print("\n" + "=" * 70)
print("PIPELINE TYPE VALIDATION")
print("=" * 70)


assert isinstance(
    train_dataset,
    tf.data.Dataset
)

assert isinstance(
    val_dataset,
    tf.data.Dataset
)

assert isinstance(
    test_dataset,
    tf.data.Dataset
)


print("\nTraining pipeline   : tf.data.Dataset ✓")
print("Validation pipeline : tf.data.Dataset ✓")
print("Testing pipeline    : tf.data.Dataset ✓")


# ==============================================================
# 17. PIPELINE DESIGN
# ==============================================================

print("\n" + "=" * 70)
print("PIPELINE DESIGN")
print("=" * 70)


print(
    """
TRAINING
--------------------------------------------------
X_train + y_train
        ↓
from_tensor_slices
        ↓
shuffle
        ↓
batch(64)
        ↓
prefetch(AUTOTUNE)
        ↓
LSTM model


VALIDATION
--------------------------------------------------
X_val + y_val
        ↓
from_tensor_slices
        ↓
batch(64)
        ↓
prefetch(AUTOTUNE)
        ↓
LSTM model


TESTING
--------------------------------------------------
X_test + y_test
        ↓
from_tensor_slices
        ↓
batch(64)
        ↓
prefetch(AUTOTUNE)
        ↓
LSTM model
"""
)


# ==============================================================
# 18. DATA LEAKAGE CHECK
# ==============================================================

print("\n" + "=" * 70)
print("DATA LEAKAGE CHECK")
print("=" * 70)


# The pipelines are constructed directly from their corresponding
# split arrays. No split is combined with another split.

print(
    "\nTraining pipeline source     : X_train / y_train"
)

print(
    "Validation pipeline source   : X_val / y_val"
)

print(
    "Testing pipeline source      : X_test / y_test"
)

print(
    "\nTraining → Validation mixing : NONE"
)

print(
    "Training → Testing mixing    : NONE"
)

print(
    "Validation → Testing mixing  : NONE"
)

print(
    "\nData leakage : NOT DETECTED"
)

# ==============================================================
# 19. FINAL PIPELINE VALIDATION
# ==============================================================

print("\n" + "=" * 70)
print("FINAL PIPELINE VALIDATION")
print("=" * 70)


print("\nTraining pipeline   : VALID")
print("Validation pipeline : VALID")
print("Testing pipeline    : VALID")

print("\nSequence modification : NONE")
print("Token modification    : NONE")
print("Data augmentation     : NONE")
print("Data leakage          : NONE")
print("Batching              : PASSED")
print("Prefetching           : PASSED")
print("Pipeline integrity    : PASSED")


# ==============================================================
# 20. FINAL SUMMARY
# ==============================================================

print("\n" + "=" * 70)
print("STEP 8 SUMMARY")
print("=" * 70)


print(
    f"""
Sequence length       : {SEQ_LEN}
Vocabulary size       : {VOCAB_SIZE}

Training examples     : {len(X_train):,}
Validation examples   : {len(X_val):,}
Testing examples      : {len(X_test):,}

Batch size            : {BATCH_SIZE}
Shuffle buffer        : {SHUFFLE_BUFFER}
Random seed            : {SEED}

Training batches      : {actual_train_batches}
Validation batches    : {actual_val_batches}
Testing batches       : {actual_test_batches}

Training shuffle      : ENABLED
Validation shuffle    : DISABLED
Testing shuffle       : DISABLED

Prefetch              : AUTOTUNE
Drop remainder        : False

Data modification     : NONE
Data leakage          : NONE DETECTED
Pipeline integrity    : PASSED
"""
)


print("=" * 70)
print("✅ STEP 8 COMPLETED — V7 DATASET PIPELINE READY")
print("=" * 70)


print("\nVariables created:")
print("train_dataset")
print("val_dataset")
print("test_dataset")


print("\nNext step:")
print("STEP 9 — V7 LSTM MODEL ARCHITECTURE")

STEP 8 — V7 DATASET PIPELINE / INPUT PIPELINE PREPARATION

REQUIRED VARIABLES
----------------------------------------------------------------------
X_train         : FOUND
y_train         : FOUND
X_val           : FOUND
y_val           : FOUND
X_test          : FOUND
y_test          : FOUND
VOCAB_SIZE      : FOUND
PAD_ID          : FOUND
OOV_ID          : FOUND
SEQ_LEN         : FOUND
word_to_id      : FOUND
id_to_word      : FOUND

PIPELINE CONFIGURATION

Sequence length : 20
Vocabulary size : 4119
Batch size      : 64
Shuffle buffer  : 10000
Random seed     : 42

Configuration validation : PASSED

NUMPY DATA VALIDATION
X_train    : shape=(23739, 20)        dtype=int32
y_train    : shape=(23739,)           dtype=int32
X_val      : shape=(2949, 20)         dtype=int32
y_val      : shape=(2949,)            dtype=int32
X_test     : shape=(2951, 20)         dtype=int32
y_test     : shape=(2951,)            dtype=int32

NumPy data validation : PASSED

SHAPE VALIDATION

Training:
X_train :

In [9]:
# ==============================================================
# STEP 9 — V7 LSTM MODEL ARCHITECTURE
# ==============================================================
# IMPORTANT:
# This step ONLY creates and validates the V7 LSTM architecture.
#
# No training is performed here.
# No model fitting is performed here.
# No checkpoint is created here.
#
# V7 architecture:
#
# Input Sequence
#       ↓
# Embedding
#       ↓
# LSTM
#       ↓
# Dense Softmax
#       ↓
# Next-token probabilities
# ==============================================================

import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

print("=" * 70)
print("STEP 9 — V7 LSTM MODEL ARCHITECTURE")
print("=" * 70)


# ==============================================================
# 1. REQUIRED VARIABLE VALIDATION
# ==============================================================

required_variables = [
    "X_train",
    "y_train",
    "X_val",
    "y_val",
    "X_test",
    "y_test",
    "VOCAB_SIZE",
    "PAD_ID",
    "OOV_ID",
    "SEQ_LEN",
    "BATCH_SIZE",
    "EMBEDDING_DIM",
    "LSTM_UNITS",
    "DROPOUT",
    "SEED"
]

missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise NameError(
        f"Missing variables: {missing_variables}\n\n"
        "Please run V7 Steps 1–8 before running Step 9."
    )

print("\nREQUIRED VARIABLES")
print("-" * 70)

for name in required_variables:
    print(f"{name:15} : FOUND")


# ==============================================================
# 2. ARCHITECTURE CONFIGURATION
# ==============================================================

print("\n" + "=" * 70)
print("ARCHITECTURE CONFIGURATION")
print("=" * 70)

print(
    f"""
Vocabulary size       : {VOCAB_SIZE}
Sequence length       : {SEQ_LEN}

Embedding dimension   : {EMBEDDING_DIM}
LSTM units            : {LSTM_UNITS}
LSTM dropout          : {DROPOUT}

PAD ID                : {PAD_ID}
OOV ID                : {OOV_ID}

Architecture:
    Input
      ↓
    Embedding
      ↓
    LSTM
      ↓
    Dense Softmax
"""
)


# ==============================================================
# 3. CONFIGURATION VALIDATION
# ==============================================================

print("=" * 70)
print("ARCHITECTURE CONFIGURATION VALIDATION")
print("=" * 70)

assert isinstance(VOCAB_SIZE, int)
assert VOCAB_SIZE > 2

assert isinstance(SEQ_LEN, int)
assert SEQ_LEN > 0

assert isinstance(EMBEDDING_DIM, int)
assert EMBEDDING_DIM > 0

assert isinstance(LSTM_UNITS, int)
assert LSTM_UNITS > 0

assert 0.0 <= DROPOUT < 1.0

assert PAD_ID == 0
assert OOV_ID == 1

print("\nAll architecture configuration checks passed.")


# ==============================================================
# 4. INPUT DATA VALIDATION
# ==============================================================

print("\n" + "=" * 70)
print("INPUT DATA VALIDATION")
print("=" * 70)

print(
    f"""
X_train shape : {X_train.shape}
y_train shape : {y_train.shape}

X_val shape   : {X_val.shape}
y_val shape   : {y_val.shape}

X_test shape  : {X_test.shape}
y_test shape  : {y_test.shape}
"""
)

assert X_train.ndim == 2
assert X_val.ndim == 2
assert X_test.ndim == 2

assert X_train.shape[1] == SEQ_LEN
assert X_val.shape[1] == SEQ_LEN
assert X_test.shape[1] == SEQ_LEN

assert len(X_train) == len(y_train)
assert len(X_val) == len(y_val)
assert len(X_test) == len(y_test)

print("Input data validation : PASSED")


# ==============================================================
# 5. TOKEN ID RANGE VALIDATION
# ==============================================================

print("\n" + "=" * 70)
print("TOKEN ID RANGE VALIDATION")
print("=" * 70)

def count_invalid_ids(array, vocab_size):
    return int(
        np.sum(
            (array < 0) |
            (array >= vocab_size)
        )
    )


train_x_invalid = count_invalid_ids(
    X_train,
    VOCAB_SIZE
)

train_y_invalid = count_invalid_ids(
    y_train,
    VOCAB_SIZE
)

val_x_invalid = count_invalid_ids(
    X_val,
    VOCAB_SIZE
)

val_y_invalid = count_invalid_ids(
    y_val,
    VOCAB_SIZE
)

test_x_invalid = count_invalid_ids(
    X_test,
    VOCAB_SIZE
)

test_y_invalid = count_invalid_ids(
    y_test,
    VOCAB_SIZE
)

print(
    f"Training     X invalid : {train_x_invalid}"
)

print(
    f"Training     y invalid : {train_y_invalid}"
)

print(
    f"Validation   X invalid : {val_x_invalid}"
)

print(
    f"Validation   y invalid : {val_y_invalid}"
)

print(
    f"Testing      X invalid : {test_x_invalid}"
)

print(
    f"Testing      y invalid : {test_y_invalid}"
)

assert train_x_invalid == 0
assert train_y_invalid == 0
assert val_x_invalid == 0
assert val_y_invalid == 0
assert test_x_invalid == 0
assert test_y_invalid == 0

print("\nToken ID range validation : PASSED")


# ==============================================================
# 6. SET RANDOM SEEDS
# ==============================================================

print("\n" + "=" * 70)
print("REPRODUCIBILITY SETUP")
print("=" * 70)

np.random.seed(SEED)
tf.random.set_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
    deterministic_status = "ENABLED"
except Exception:
    deterministic_status = "NOT AVAILABLE"

print(f"\nRandom seed          : {SEED}")
print(f"TensorFlow seed      : {SEED}")
print(f"Deterministic ops    : {deterministic_status}")


# ==============================================================
# 7. BUILD V7 MODEL
# ==============================================================

print("\n" + "=" * 70)
print("BUILDING V7 LSTM MODEL")
print("=" * 70)

lstm_v7 = Sequential(
    [
        Embedding(
            input_dim=VOCAB_SIZE,
            output_dim=EMBEDDING_DIM,
            input_length=SEQ_LEN,
            name="embedding_v7"
        ),

        LSTM(
            units=LSTM_UNITS,
            dropout=DROPOUT,
            return_sequences=False,
            name="lstm_v7"
        ),

        Dense(
            VOCAB_SIZE,
            activation="softmax",
            name="output_v7"
        )
    ],
    name="lstm_v7"
)


# ==============================================================
# 8. BUILD MODEL
# ==============================================================

lstm_v7.build(
    input_shape=(None, SEQ_LEN)
)

print("\nModel built successfully.")


# ==============================================================
# 9. MODEL SUMMARY
# ==============================================================

print("\n" + "=" * 70)
print("V7 MODEL SUMMARY")
print("=" * 70)

lstm_v7.summary()


# ==============================================================
# 10. LAYER VALIDATION
# ==============================================================

print("\n" + "=" * 70)
print("LAYER VALIDATION")
print("=" * 70)

layers = lstm_v7.layers

print(
    f"\nNumber of layers : {len(layers)}"
)

for index, layer in enumerate(
    layers,
    start=1
):
    print(
        f"{index}. "
        f"{layer.name:20} "
        f"{layer.__class__.__name__}"
    )

assert len(layers) == 3

assert isinstance(
    layers[0],
    Embedding
)

assert isinstance(
    layers[1],
    LSTM
)

assert isinstance(
    layers[2],
    Dense
)

print("\nLayer structure : VALID")


# ==============================================================
# 11. EMBEDDING VALIDATION
# ==============================================================

print("\n" + "=" * 70)
print("EMBEDDING LAYER VALIDATION")
print("=" * 70)

embedding_layer = lstm_v7.get_layer(
    "embedding_v7"
)

print(
    f"\nInput dimension  : "
    f"{embedding_layer.input_dim}"
)

print(
    f"Output dimension : "
    f"{embedding_layer.output_dim}"
)

assert embedding_layer.input_dim == VOCAB_SIZE
assert embedding_layer.output_dim == EMBEDDING_DIM

print("\nEmbedding configuration : VALID")


# ==============================================================
# 12. LSTM VALIDATION
# ==============================================================

print("\n" + "=" * 70)
print("LSTM LAYER VALIDATION")
print("=" * 70)

lstm_layer = lstm_v7.get_layer(
    "lstm_v7"
)

print(
    f"\nUnits             : "
    f"{lstm_layer.units}"
)

print(
    f"Dropout           : "
    f"{lstm_layer.dropout}"
)

print(
    f"Return sequences  : "
    f"{lstm_layer.return_sequences}"
)

assert lstm_layer.units == LSTM_UNITS

assert np.isclose(
    lstm_layer.dropout,
    DROPOUT
)

assert lstm_layer.return_sequences is False

print("\nLSTM configuration : VALID")


# ==============================================================
# 13. OUTPUT LAYER VALIDATION
# ==============================================================

print("\n" + "=" * 70)
print("OUTPUT LAYER VALIDATION")
print("=" * 70)

output_layer = lstm_v7.get_layer(
    "output_v7"
)

print(
    f"\nOutput units      : "
    f"{output_layer.units}"
)

print(
    f"Activation        : "
    f"{output_layer.activation.__name__}"
)

assert output_layer.units == VOCAB_SIZE

assert (
    output_layer.activation.__name__
    == "softmax"
)

print("\nOutput layer configuration : VALID")


# ==============================================================
# 14. OUTPUT SHAPE VALIDATION
# ==============================================================

print("\n" + "=" * 70)
print("OUTPUT SHAPE VALIDATION")
print("=" * 70)

expected_output_shape = (
    None,
    VOCAB_SIZE
)

actual_output_shape = (
    lstm_v7.output_shape
)

print(
    f"\nExpected output shape : "
    f"{expected_output_shape}"
)

print(
    f"Actual output shape   : "
    f"{actual_output_shape}"
)

assert actual_output_shape == expected_output_shape

print("\nOutput shape : VALID")


# ==============================================================
# 15. PARAMETER COUNT
# ==============================================================

print("\n" + "=" * 70)
print("PARAMETER COUNT")
print("=" * 70)

total_params = lstm_v7.count_params()

trainable_params = np.sum(
    [
        np.prod(variable.shape)
        for variable in lstm_v7.trainable_variables
    ]
)

non_trainable_params = np.sum(
    [
        np.prod(variable.shape)
        for variable in lstm_v7.non_trainable_variables
    ]
)

print(
    f"\nTotal parameters       : "
    f"{total_params:,}"
)

print(
    f"Trainable parameters   : "
    f"{trainable_params:,}"
)

print(
    f"Non-trainable params   : "
    f"{non_trainable_params:,}"
)

assert total_params == (
    trainable_params +
    non_trainable_params
)

print("\nParameter count validation : PASSED")


# ==============================================================
# 16. FORWARD PASS TEST
# ==============================================================

print("\n" + "=" * 70)
print("FORWARD PASS TEST")
print("=" * 70)

sample_input = X_train[:4]

print(
    f"\nSample input shape : "
    f"{sample_input.shape}"
)

sample_output = lstm_v7(
    sample_input,
    training=False
)

sample_output_np = (
    sample_output.numpy()
)

print(
    f"Sample output shape : "
    f"{sample_output_np.shape}"
)

expected_sample_shape = (
    4,
    VOCAB_SIZE
)

assert (
    sample_output_np.shape
    == expected_sample_shape
)

print(
    "Expected output shape : "
    f"{expected_sample_shape}"
)

print("\nForward pass : PASSED")


# ==============================================================
# 17. PROBABILITY VALIDATION
# ==============================================================

print("\n" + "=" * 70)
print("PROBABILITY OUTPUT VALIDATION")
print("=" * 70)

probability_sums = np.sum(
    sample_output_np,
    axis=1
)

print(
    "\nProbability sums:"
)

for i, value in enumerate(
    probability_sums,
    start=1
):
    print(
        f"Sample {i} : {value:.6f}"
    )

assert np.allclose(
    probability_sums,
    1.0,
    atol=1e-5
)

assert np.all(
    sample_output_np >= 0
)

assert np.all(
    sample_output_np <= 1
)

print(
    "\nSoftmax probability validation : PASSED"
)


# ==============================================================
# 18. PREDICTION ID VALIDATION
# ==============================================================

print("\n" + "=" * 70)
print("PREDICTION ID VALIDATION")
print("=" * 70)

sample_predicted_ids = np.argmax(
    sample_output_np,
    axis=1
)

print(
    f"\nPredicted IDs : "
    f"{sample_predicted_ids.tolist()}"
)

assert np.all(
    sample_predicted_ids >= 0
)

assert np.all(
    sample_predicted_ids < VOCAB_SIZE
)

print("Prediction ID range : VALID")


# ==============================================================
# 19. MODEL INPUT / OUTPUT CONTRACT
# ==============================================================

print("\n" + "=" * 70)
print("MODEL INPUT / OUTPUT CONTRACT")
print("=" * 70)

print(
    f"""
INPUT
--------------------------------------------------
Shape:
    (batch_size, {SEQ_LEN})

Datatype:
    integer token IDs

Valid token IDs:
    {PAD_ID} → {VOCAB_SIZE - 1}


OUTPUT
--------------------------------------------------
Shape:
    (batch_size, {VOCAB_SIZE})

Datatype:
    floating-point probabilities

Output meaning:
    Probability of every possible next token
"""
)

print("Model input/output contract : VALID")


# ==============================================================
# 20. ARCHITECTURE FLOW
# ==============================================================

print("\n" + "=" * 70)
print("V7 ARCHITECTURE FLOW")
print("=" * 70)

print(
    """
Token IDs
    │
    │ (batch, 20)
    ▼
Embedding
    │
    │ (batch, 20, 128)
    ▼
LSTM
    │
    │ (batch, 128)
    ▼
Dense
    │
    │ (batch, 4119)
    ▼
Softmax
    │
    │ probability distribution
    ▼
Next-token prediction
"""
)


# ==============================================================
# 21. FINAL ARCHITECTURE VALIDATION
# ==============================================================

print("\n" + "=" * 70)
print("FINAL ARCHITECTURE VALIDATION")
print("=" * 70)

assert lstm_v7.name == "lstm_v7"

assert len(lstm_v7.layers) == 3

assert (
    lstm_v7.layers[0].__class__.__name__
    == "Embedding"
)

assert (
    lstm_v7.layers[1].__class__.__name__
    == "LSTM"
)

assert (
    lstm_v7.layers[2].__class__.__name__
    == "Dense"
)

assert (
    lstm_v7.output_shape[-1]
    == VOCAB_SIZE
)

assert total_params > 0

print(
    "\nAll final architecture checks passed."
)


# ==============================================================
# 22. STEP 9 SUMMARY
# ==============================================================

print("\n" + "=" * 70)
print("STEP 9 SUMMARY")
print("=" * 70)

print(
    f"""
Model name            : lstm_v7

Vocabulary size       : {VOCAB_SIZE}
Sequence length       : {SEQ_LEN}

Embedding dimension   : {EMBEDDING_DIM}
LSTM units            : {LSTM_UNITS}
Dropout               : {DROPOUT}

Number of layers      : {len(lstm_v7.layers)}

Total parameters      : {total_params:,}
Trainable parameters  : {trainable_params:,}

Input shape           : (batch, {SEQ_LEN})
Output shape          : (batch, {VOCAB_SIZE})

Architecture          :
    Embedding
        ↓
    LSTM
        ↓
    Dense + Softmax

Forward pass          : PASSED
Probability check     : PASSED
Output shape          : PASSED
Token range           : PASSED
Architecture          : VALID
"""
)

print("=" * 70)
print("✅ STEP 9 COMPLETED — V7 LSTM ARCHITECTURE READY")
print("=" * 70)

print(
    """
Variables created:
lstm_v7

Next step:
STEP 10 — V7 MODEL COMPILATION
"""
)

STEP 9 — V7 LSTM MODEL ARCHITECTURE


NameError: Missing variables: ['DROPOUT']

Please run V7 Steps 1–8 before running Step 9.